# LLaVA-Med 7B 재현 — VQA-RAD · SLAKE (한 노트북)

논문: *LLaVA-Med: Training a Large Language-and-Vision Assistant for Biomedicine in One Day* (arXiv:2306.00890). Expert-CFG 논문 Table 1 의 LLaVA-Med 행을 재현한다. 위쪽(공통 준비)을 한 번 실행한 뒤 **A(VQA-RAD)와 B(SLAKE)를 각각 독립적으로** 실행할 수 있다.

| 데이터셋 | 방식 | 재현 목표 (Open / Closed / Overall) |
|---|---|---|
| VQA-RAD | 공식 파인튜닝 delta 를 LLaMA-7B 에 병합, 학습 없음 | 61.52 / 84.19 / 75.19 |
| SLAKE | 공식 SLAKE delta 에 가중치가 없어(README 만 존재) **QLoRA 근사**: 의료 instruct 모델에서 SLAKE train 으로 학습 | 83.08 / 85.34 / 83.97 |

SLAKE 는 논문(전체 파인튜닝)과 **같은 조건이 아니므로** 결과표에 "QLoRA 근사"로 표시한다.

| 구간 | 내용 |
|---|---|
| 공통 | 공통-1 환경 → 공통-2 정의 · 채점 함수 → 공통-3 transformers 고정 → 공통-4 vqa_scoring.py |
| **A. VQA-RAD** | 병합 → test 451문항 준비 → greedy 추론(closed 접미어) → 채점 · 논문 비교 |
| **B. SLAKE** | instruct 모델 병합 → 학습 · val 로 학습률/epoch 선택 → test 는 선택 후 한 번 → 최종 표 · 곡선 |
| **C. 종합** | 두 데이터셋의 Open / Closed / ALL 을 원본 채점과 S1+S2 로 나란히 + 논문 대비 격차 + 신뢰구간 |
| 부록 | A: VQA-RAD closed 프롬프트 변형 · 편향 보정(규칙 완화 행) / B: SLAKE yes/no 편향 보정(효과 없음) · 프로젝터 반영 확인 |

**규칙**: train 만 학습, val 로 학습률·epoch 선택, test 는 마지막에 한 번. VQA-RAD 기준 행은 test 로 고른 설정이 없다(공식 delta + 접미어 + greedy). 부록 A 는 test 로 규칙을 고른 탐색이므로 **규칙 완화 행으로 따로 표기**한다. SLAKE 의 `FINAL_TRAINVAL = False`(기본)는 규칙 준수 행, `True` 는 train+val 학습이므로 규칙 완화 행이다.

> 이 노트북은 `LLaVA_Med_VQA_RAD_reproduction.ipynb` 와 `LLaVA_Med_SLAKE_QLoRA_reproduction.ipynb` 를 합친 것이다(셀 내용은 그대로, 중복 준비 셀만 정리). 진단·변형 셀의 전체 이력은 분리 전 원본 `LLaVA_Med_expert_cfg_reproduction.ipynb` 에 남아 있다.


## 공통 준비

In [13]:
# 공통-1. 환경 준비 (Drive · 레포 v1.0.0 · 추론 venv · LLaMA-7B 베이스 · greedy 패치). 이미 되어 있는 것은 건너뛰므로 다시 실행해도 된다.
import os, re, json, subprocess, glob, math
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/expert_cfg_llavamed'      # 앞 절과 같은 폴더 (결과 파일이 모인다)
EVAL_DIR   = '/content/llava_eval_data'
PY         = '/content/venv_llava/bin/python'                  # 추론용 환경 (LLaVA-Med v1.0, torch 2.0.0)
for d in (DRIVE_ROOT, EVAL_DIR):
    os.makedirs(d, exist_ok=True)
def tee(cmd, log=None):
    # 명령의 출력을 한 줄씩 셀에 보여 주고(진행 상황), log 가 있으면 파일에도 저장한다. 종료 코드를 돌려준다.
    f = open(log, 'a') if log else None
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        if f:
            f.write(line); f.flush()
    rc = p.wait()
    if f:
        f.close()
    return rc

def sh(cmd):
    rc = tee(cmd)
    if rc != 0:
        raise RuntimeError(f'실패(종료 코드 {rc}): {cmd}')

# 1) 레포 (v1.0.0 태그)
if not os.path.isdir('/content/LLaVA-Med'):
    sh('git clone https://github.com/microsoft/LLaVA-Med.git /content/LLaVA-Med')
sh('cd /content/LLaVA-Med && git checkout v1.0.0')
os.chdir('/content/LLaVA-Med')

# 2) 추론용 venv — 파일이 있는지가 아니라 실제로 import 되는지로 점검하고, 안 되면 (재)설치한다
def venv_ok():
    if not os.path.exists(PY):
        return False, 'venv 없음'
    r = subprocess.run([PY, '-c', 'import torch, transformers, open_clip, einops, shortuuid, llava'],
                       capture_output=True, text=True, cwd='/content/LLaVA-Med')
    return r.returncode == 0, r.stderr.strip().splitlines()[-1] if r.stderr.strip() else ''

ok, why = venv_ok()
if not ok:
    print('추론 venv 점검 실패 →', why, '→ (재)설치합니다')
    sh('pip install -q uv')
    open('/content/constraints_llava.txt', 'w').write('torch==2.0.0\ntorchvision==0.15.1\n')   # 다른 패키지가 torch 를 올리지 못하게 고정
    if not os.path.exists(PY):
        sh('uv venv /content/venv_llava --python 3.10')
    CST = '-c /content/constraints_llava.txt'
    sh(f'uv pip install --python /content/venv_llava/bin/python {CST} torch==2.0.0 torchvision==0.15.1')
    sh(f'cd /content/LLaVA-Med && uv pip install --python /content/venv_llava/bin/python {CST} -e .')
    sh(f'uv pip install --python /content/venv_llava/bin/python {CST} einops shortuuid ninja open-clip-torch==2.24.0 "numpy<2"')
    ok, why = venv_ok()
    assert ok, f'설치 후에도 import 실패: {why}'
print(subprocess.run([PY, '-c', 'import torch, transformers, open_clip, llava; print(torch.__version__, transformers.__version__, torch.cuda.is_available())'],
                     capture_output=True, text=True, cwd='/content/LLaVA-Med').stdout.strip())

# 3) LLaMA-7B 베이스 (Drive 사본 우선, 없으면 HF)
from huggingface_hub import snapshot_download
LLAMA_BASE = '/content/drive/MyDrive/llama-7b-hf'
if not os.path.isdir(LLAMA_BASE):
    LLAMA_BASE = snapshot_download('huggyllama/llama-7b', local_dir='/content/llama-7b-hf', allow_patterns=['*.json', '*.model', '*.safetensors'])
assert os.path.exists(os.path.join(LLAMA_BASE, 'config.json')), 'LLaMA-7B 가중치가 올바르지 않습니다'
print('LLAMA_BASE =', LLAMA_BASE)

# 4) 추론 스크립트를 greedy 로 패치 (VQA-RAD 평가와 같은 조건)
p = '/content/LLaVA-Med/llava/eval/model_vqa_med.py'
s = open(p).read()
pat = r"do_sample=True,\s*temperature=0\.7,\s*max_new_tokens=1024,"
if re.search(pat, s):
    open(p, 'w').write(re.sub(pat, "do_sample=False,\n                max_new_tokens=1024,", s, count=1)); print('greedy 패치 완료')
else:
    assert 'do_sample=False,' in s, '패치 대상 문자열을 찾지 못했습니다'; print('greedy 패치 이미 적용됨')

Cloning into '/content/LLaVA-Med'...
Filtering content:  20% (2/10)
Filtering content:  30% (3/10)
Filtering content:  40% (4/10)
Filtering content:  50% (5/10)
Filtering content:  60% (6/10), 19.65 MiB | 3.04 MiB/s
Filtering content:  60% (6/10), 104.30 MiB | 3.61 MiB/s
Filtering content:  70% (7/10), 104.30 MiB | 3.61 MiB/s
Filtering content:  80% (8/10), 187.90 MiB | 6.21 MiB/s
Filtering content:  80% (8/10), 310.72 MiB | 6.75 MiB/s
Filtering content:  90% (9/10), 310.72 MiB | 6.75 MiB/s
Filtering content: 100% (10/10), 472.11 MiB | 8.05 MiB/s
Filtering content: 100% (10/10), 813.63 MiB | 6.73 MiB/s
Filtering content: 100% (10/10), 813.63 MiB | 6.66 MiB/s, done.
Note: switching to 'v1.0.0'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

LLAMA_BASE = /content/llama-7b-hf
greedy 패치 완료


In [14]:
from google.colab import drive; drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/expert_cfg_llavamed'; EVAL_DIR = '/content/llava_eval_data'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# 공통-2. 공통 정의 — 질문 형식(closed 접미어), 평가 파일 작성, 채점 함수 (두 데이터셋 공용)
import zipfile
from collections import Counter
import numpy as np
from huggingface_hub import hf_hub_download


# closed 질문에 붙이는 접미어 (학습·평가 모두 동일)
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'

def write_eval_files(name, rows, get_image):
    """model_vqa_med.py 입력(JSON 배열 + conversations)과 채점용 참조 파일을 EVAL_DIR 에 만든다."""
    questions, refs = [], {}
    for i, r in enumerate(rows):
        questions.append({'id': i, 'image': get_image(r),
                          'conversations': [{'from': 'human', 'value': r['question'] + (CLOSED_SUFFIX if r['answer_type'] == 'CLOSED' else '')},
                                            {'from': 'gpt', 'value': str(r['answer'])}]})
        refs[i] = {'answer': str(r['answer']), 'answer_type': r['answer_type']}
    json.dump(questions, open(f'{EVAL_DIR}/{name}_test.json', 'w'))
    json.dump(refs, open(f'{EVAL_DIR}/{name}_ref.json', 'w'))
    n_open = sum(v['answer_type'] == 'OPEN' for v in refs.values())
    print(f'{name}: {len(rows)}문항 (OPEN {n_open} / CLOSED {len(rows) - n_open})')

# 채점 (기존 방식): closed = 정답 단어가 예측 토큰에 포함되면 정답, open = 토큰 recall
def normalize(text):
    text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()

def closed_accuracy(pred, gt):
    return float(gt.strip().lower() in normalize(pred))

def open_recall(pred, gt):
    gt_toks, pred_toks = Counter(normalize(gt)), Counter(normalize(pred))
    if not gt_toks:
        return 0.0
    return sum((gt_toks & pred_toks).values()) / sum(gt_toks.values())

def score_jsonl(ref_file, pred_file):
    refs = {int(k): v for k, v in json.load(open(ref_file)).items()}
    preds = {}
    for line in open(pred_file):
        p = json.loads(line)
        preds[int(p['question_id'])] = p['text']
    assert set(preds) == set(refs), f'예측 {len(preds)}개 / 참조 {len(refs)}개 — 추론이 끝나지 않았거나 id 불일치'
    cl, op = [], []
    for i, ref in refs.items():
        if ref['answer_type'] == 'CLOSED':
            cl.append(closed_accuracy(preds[i], ref['answer']))
        else:
            op.append(open_recall(preds[i], ref['answer']))
    return {'open': 100 * np.mean(op), 'closed': 100 * np.mean(cl), 'overall': 100 * np.mean(cl + op), 'n_open': len(op), 'n_closed': len(cl)}
print('준비 완료')

준비 완료


In [ ]:
# 공통-3. transformers 를 LLaVA-Med v1.0 이 요구하는 커밋으로 고정 (이미 맞으면 바로 끝난다)
!uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt "transformers @ git+https://github.com/huggingface/transformers@cae78c46" tokenizers==0.12.1 "numpy<2"
!cd /content/LLaVA-Med && /content/venv_llava/bin/python -c "import torch, transformers, llava; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"


In [9]:
# 공통-4. 보조 채점·신뢰구간 모듈(vqa_scoring.py) 받기 — 결과 셀들이 선택적으로 사용한다
import os, sys, subprocess
if not os.path.isdir('/content/Expert-CFG'):
    subprocess.run(['git', 'clone', '-b', 'VQA_RAD_reproduction', 'https://github.com/MinseongKim-dev/Expert-CFG', '/content/Expert-CFG'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Expert-CFG', 'pull'])
if '/content/Expert-CFG' not in sys.path: sys.path.insert(0, '/content/Expert-CFG')
print('vqa_scoring.py 있음:', os.path.exists('/content/Expert-CFG/vqa_scoring.py'))


vqa_scoring.py 있음: True


## A. VQA-RAD — 공식 delta 병합 · 기준 행 (학습 없음)

공식 파인튜닝 delta(`katielink/llava-med-7b-vqarad-delta`)를 LLaMA-7B 에 병합해 평가한다. closed 질문에는 ` Please choose from the following two options: [yes, no]` 접미어를 붙이고(레포 샘플 프롬프트 형식), 디코딩은 greedy 다(레포 기본은 샘플링 T=0.7 이라 공통-1 에서 패치).

In [16]:
from huggingface_hub import snapshot_download

VQARAD_DELTA = snapshot_download('katielink/llava-med-7b-vqarad-delta', local_dir='/content/delta_vqarad')
print('VQARAD_DELTA =', VQARAD_DELTA)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

VQARAD_DELTA = /content/delta_vqarad


In [ ]:
%cd /content/LLaVA-Med
!{PY} -m llava.model.apply_delta \
    --base-model-path {LLAMA_BASE} \
    --target-model-path /content/llava-med-7b-vqarad \
    --delta-path {VQARAD_DELTA}
assert os.path.exists('/content/llava-med-7b-vqarad/config.json'), '병합 실패'

### A-1. 평가 데이터 (VQA-RAD test 451문항: open 179 / closed 272)

In [10]:
import os, json, shutil, subprocess

EVAL_DIR = '/content/llava_eval_data'
os.makedirs(EVAL_DIR, exist_ok=True)

_GDRIVE_JSON   = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
_GDRIVE_IMGDIR = '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder'

def answer_type_of(ans):
    return 'CLOSED' if str(ans).strip().lower() in ('yes', 'no') else 'OPEN'

# LLaVA-Med 레포의 closed(yes/no) 문항 프롬프트 형식 (llava/eval/eval_metrics/answer-file.jsonl 의 prompt 필드).
# 파인튜닝 데이터가 이 형식이었다면 접미어 없이 물으면 yes/no 로 답하지 않아 closed 점수가 크게 떨어진다.
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'

def write_eval_files(name, rows, get_image):
    """rows: dict(question, answer, answer_type, ...). 질문 JSON 배열 + 참조 JSON 저장."""
    questions, refs = [], {}
    for i, r in enumerate(rows):
        questions.append({
            'id': i,
            'image': get_image(r),
            'conversations': [{'from': 'human', 'value': r['question'] + (CLOSED_SUFFIX if r['answer_type'] == 'CLOSED' else '')},
                              {'from': 'gpt',   'value': str(r['answer'])}],
        })
        refs[i] = {'answer': str(r['answer']), 'answer_type': r['answer_type']}
    json.dump(questions, open(f'{EVAL_DIR}/{name}_test.json', 'w'))
    json.dump(refs, open(f'{EVAL_DIR}/{name}_ref.json', 'w'))
    n_open = sum(v['answer_type'] == 'OPEN' for v in refs.values())
    print(f'{name}: {len(rows)}문항 (OPEN {n_open} / CLOSED {len(rows) - n_open})')

# ── VQA-RAD ──
_raw_json = _GDRIVE_JSON
if not os.path.exists(_raw_json):
    _raw_json = os.path.join(EVAL_DIR, 'VQA_RAD_Dataset_Public.json')
    subprocess.run(['wget', '-q', '-O', _raw_json, 'https://osf.io/download/9dpze/'], timeout=120, check=True)
all_data = json.load(open(_raw_json))
for item in all_data:
    item['answer_type'] = item.get('answer_type', answer_type_of(item.get('answer', ''))).upper()
_TEST_TYPES = {'test_freeform', 'test_para'}
assert _TEST_TYPES & {str(r.get('phrase_type', '')).lower() for r in all_data}, 'phrase_type 에 test 표시가 없습니다'
vqarad_test = [r for r in all_data if str(r.get('phrase_type', '')).lower() in _TEST_TYPES]
assert len(vqarad_test) == 451, len(vqarad_test)

VQARAD_IMG_DIR = _GDRIVE_IMGDIR
assert os.path.isdir(VQARAD_IMG_DIR), f'이미지 폴더 없음: {VQARAD_IMG_DIR}'
write_eval_files('vqarad', vqarad_test, lambda r: r['image_name'])
assert all(os.path.exists(os.path.join(VQARAD_IMG_DIR, r['image_name'])) for r in vqarad_test), '이미지 누락'

vqarad: 451문항 (OPEN 179 / CLOSED 272)


In [11]:
# 2) 전체 451문항 질문 파일 (closed 에는 접미어) 생성
import json
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'
qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))
ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
for q in qs:
    v = q['conversations'][0]['value']
    if ref[str(q['id'])]['answer_type'] == 'CLOSED' and not v.endswith(CLOSED_SUFFIX):
        q['conversations'][0]['value'] = v + CLOSED_SUFFIX
json.dump(qs, open(f'{EVAL_DIR}/vqarad_test_suffix.json', 'w'))
print(len(qs), '|', sum(q['conversations'][0]['value'].endswith(CLOSED_SUFFIX) for q in qs), '개 접미어 적용')   # 451 | 272


451 | 272 개 접미어 적용


In [12]:
# 3) greedy 추론 (백그라운드)
%cd /content/LLaVA-Med
!PYTHONUNBUFFERED=1 nohup {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_test_suffix.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {DRIVE_ROOT}/vqarad_pred_greedy.jsonl \
    > /content/vqarad_greedy.log 2>&1 &


[Errno 2] No such file or directory: '/content/LLaVA-Med'
/content


In [ ]:
import os
_pred = f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'
n = sum(1 for _ in open(_pred)) if os.path.exists(_pred) else 0
print(f'{n}/451 완료')
!tail -c 400 /content/vqarad_greedy.log | tr '\r' '\n' | tail -n 3
!pgrep -f model_vqa_med.py > /dev/null && echo "추론 실행 중" || echo "추론 프로세스 없음"


### A-2. 채점 · 논문 비교

In [7]:
# 5-2. 기준 행 결과 — 공식 delta, closed 접미어, greedy (test 로 고른 설정 없음) + 논문 비교 + 보조 채점(S1+S2) + 부트스트랩 구간
import json, os, sys
_pred = f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'
n = sum(1 for _ in open(_pred))
assert n == 451, f'예측 {n}/451 — 추론이 아직 끝나지 않았습니다'
first = [json.loads(l) for l in open(_pred)][:3]
print('prompt 확인:', [p['prompt'][-70:] for p in first])      # closed 문항은 끝에 접미어가 보여야 한다

PAPER = {'open': 61.52, 'closed': 84.19, 'overall': 75.19}      # 논문 Table 1, LLaVA-Med VQA-RAD
m = score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', _pred)
print(f"\n{'':12s}{'open':>8s}{'closed':>8s}{'overall':>9s}")
print(f"{'논문':10s}{PAPER['open']:8.2f}{PAPER['closed']:8.2f}{PAPER['overall']:9.2f}")
print(f"{'기준 행':9s}{m['open']:8.2f}{m['closed']:8.2f}{m['overall']:9.2f}")
print(f"{'격차':10s}{m['open']-PAPER['open']:+8.2f}{m['closed']-PAPER['closed']:+8.2f}{m['overall']-PAPER['overall']:+9.2f}")

# 보조 채점(S1+S2)과 신뢰구간: 공용 모듈 vqa_scoring.py 가 Drive 의 expert_cfg 폴더나 clone 한 레포에 있을 때만 실행한다
for _p in ('/content/drive/MyDrive/expert_cfg', '/content/Expert-CFG'):
    if os.path.isdir(_p) and _p not in sys.path: sys.path.insert(0, _p)
try:
    import vqa_scoring as vs
    R = vs.load_llava(f'{EVAL_DIR}/vqarad_ref.json', _pred)
    b = vs.summarize(R, vs.score_s1s2)
    print(f"{'+ S1+S2':9s}{b['open']:8.2f}{b['closed']:8.2f}{b['overall']:9.2f}   (별도 행: 규칙이 Phi 의 test 오답을 보고 만들어졌음)")
    print()
    vs.print_ci(R, PAPER, 'LLaVA-Med VQA-RAD 기준 행', vs.score_phi)
except ImportError:
    print('\n(vqa_scoring.py 를 찾지 못해 보조 채점·신뢰구간은 건너뜀 — Drive 의 expert_cfg 폴더에 올리거나 레포를 clone 하세요)')


prompt 확인: ['urysm? Please choose from the following two options: [yes, no]\n<image>', ' side? Please choose from the following two options: [yes, no]\n<image>', 'plane? Please choose from the following two options: [yes, no]\n<image>']


NameError: name 'score_jsonl' is not defined

## B. SLAKE — QLoRA 근사 (공식 SLAKE delta 사용 불가)

LLaVA-Med 의료 instruct 모델(v1.0, 다운스트림 파인튜닝 전)에서 4비트(NF4) + LoRA(r=64, alpha=16)로 SLAKE 영어 train 을 학습한다. 2단계에서는 `mm_projector` 도 함께 학습한다. **val 은 선택 전용**(학습률 · epoch), **test 는 C-9b 에서 처음 읽는다.** 순서: C-1~C-3 → 스크립트 두 개 → C-4 → C-7~C-9 → (C-9c/9d) → C-9b → C-10~C-13 → C-14/C-15.

In [ ]:
# S-0. SLAKE 이미지 받기 (영어 질문은 C-3 / C-9b 에서 만든다)
import zipfile
from huggingface_hub import hf_hub_download
SLAKE_ROOT = '/content/slake_data'
if not os.path.isdir(f'{SLAKE_ROOT}/imgs'):
    with zipfile.ZipFile(hf_hub_download('BoKelvin/SLAKE', 'imgs.zip', repo_type='dataset')) as z:
        z.extractall(SLAKE_ROOT)
SLAKE_IMG_DIR = f'{SLAKE_ROOT}/imgs'


In [ ]:
# C-1. 출발점 모델: LLaVA-Med 의료 instruct 모델 (공식 SLAKE delta 는 받을 수 없음)
# apply_delta 는 LLaMA-7B(fp16 약 13.5GB)와 delta 를 동시에 메모리에 올린다 → 시스템 RAM 이 약 30GB 이상 필요하다 (Colab 고용량 RAM 런타임 권장).
# 저장 순서는 shard → index(pytorch_model.bin.index.json) → 토크나이저 이므로, 이 파일들이 모두 있고 shard 용량이 맞아야 "완전한 병합"이다.
import os, glob, shutil, json
from huggingface_hub import snapshot_download

INSTRUCT_DELTA = snapshot_download('microsoft/llava-med-7b-delta', local_dir='/content/delta_instruct')
INSTRUCT_MODEL = '/content/llava-med-7b-instruct'

def merged_ok(d):
    idx = f'{d}/pytorch_model.bin.index.json'
    if not os.path.exists(f'{d}/tokenizer.model') or not os.path.exists(f'{d}/config.json'):
        return False
    if os.path.exists(idx):
        meta = json.load(open(idx))
        files = sorted(set(meta['weight_map'].values()))
        if not all(os.path.exists(f'{d}/{f}') for f in files):
            return False
        total = sum(os.path.getsize(f'{d}/{f}') for f in files)
        return total >= 0.98 * meta['metadata']['total_size']
    return os.path.exists(f'{d}/pytorch_model.bin') and os.path.getsize(f'{d}/pytorch_model.bin') > 12e9

sh('free -g | head -2; df -h /content | tail -1')
if merged_ok(INSTRUCT_MODEL):
    print('이미 완전하게 병합돼 있습니다:', INSTRUCT_MODEL)
else:
    print('병합 결과가 없거나 불완전합니다 → 지우고 다시 병합합니다 (수 분 걸립니다. 중간에 셀을 중단하지 마세요)')
    shutil.rmtree(INSTRUCT_MODEL, ignore_errors=True)
    sh(f'cd /content/LLaVA-Med && {PY} -m llava.model.apply_delta --base-model-path {LLAMA_BASE} --target-model-path {INSTRUCT_MODEL} --delta-path {INSTRUCT_DELTA}')
assert merged_ok(INSTRUCT_MODEL), f'병합이 불완전합니다 ({INSTRUCT_MODEL}). 위 로그의 오류/Killed 여부를 확인하세요'
sh(f'ls -la {INSTRUCT_MODEL} | head -20')


In [ ]:
# C-2. 학습용 별도 venv (기존 venv_llava 는 추론용으로 그대로 둔다)
# 중요: 다른 패키지가 torch 를 올려 버리면 bitsandbytes 0.41 이 CUDA 라이브러리를 못 찾는다 → 제약 파일로 torch/torchvision 을 고정한다.
!pip install -q uv
!rm -rf /content/venv_qlora
!uv venv /content/venv_qlora --python 3.10
QPY = '/content/venv_qlora/bin/python'
open('/content/constraints_qlora.txt', 'w').write('torch==2.0.0\ntorchvision==0.15.1\n')
!uv pip install --python {QPY} -c /content/constraints_qlora.txt torch==2.0.0 torchvision==0.15.1
!uv pip install --python {QPY} -c /content/constraints_qlora.txt transformers==4.31.0 tokenizers==0.13.3 accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.41.0 scipy psutil sentencepiece protobuf==3.20.3 einops shortuuid open-clip-torch==2.24.0 pillow matplotlib "numpy<2"
!cd /content/LLaVA-Med && uv pip install --python {QPY} -c /content/constraints_qlora.txt --no-deps -e .
# 점검: torch 의 CUDA 버전이 11.x 여야 하고, bitsandbytes 가 SUCCESS 를 보여야 한다
!{QPY} -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())"
!{QPY} -m bitsandbytes 2>&1 | grep -E "SUCCESS|CUDA_VERSION|Setup Failed|Defaulting" | head
!cd /content/LLaVA-Med && {QPY} -c "import torch, transformers, peft, bitsandbytes, llava; import importlib.metadata as md; print('OK', torch.__version__, transformers.__version__, peft.__version__, md.version('bitsandbytes'))"


In [ ]:
# C-3. SLAKE 영어 train / val 생성 (질문 형식은 평가 파일과 동일: closed 에는 접미어). test 는 만들지 않는다.
import json, os
from huggingface_hub import hf_hub_download

QDATA = '/content/qlora_data'
os.makedirs(QDATA, exist_ok=True)

def build(src, out):
    rows = [r for r in json.load(open(hf_hub_download('BoKelvin/SLAKE', src, repo_type='dataset'))) if r['q_lang'] == 'en']
    res = []
    for r in rows:
        at = r['answer_type'].upper()
        res.append({'image': r['img_name'], 'question': r['question'] + (CLOSED_SUFFIX if at == 'CLOSED' else ''),
                    'answer': str(r['answer']), 'answer_type': at})
    json.dump(res, open(f'{QDATA}/{out}', 'w'))
    n_open = sum(x['answer_type'] == 'OPEN' for x in res)
    print(f'{src} -> {out}: {len(res)}문항 (OPEN {n_open} / CLOSED {len(res) - n_open})')
    assert all(os.path.exists(os.path.join(SLAKE_IMG_DIR, x['image'])) for x in res), '이미지 누락'
    return res

_tr, _va = build('train.json', 'slake_train.json'), build('validation.json', 'slake_val.json')
# train/val 겹침 점검 (이미지+질문). test 와의 점검은 test 를 처음 읽는 C-9b 에서 한다.
key = lambda r: (r['image'], r['question'])
print('train∩val:', len({key(r) for r in _tr} & {key(r) for r in _va}))
json.dump(_tr + _va, open(f'{QDATA}/slake_trainval.json', 'w'))
print('slake_trainval.json:', len(_tr) + len(_va), '문항 (train+val — 긴 학습에서 FINAL_TRAINVAL=True 일 때 사용)')


In [ ]:
%%writefile /content/LLaVA-Med/train_qlora_slake.py
"""LLaVA-Med v1.0 (LLaMA-7B) 를 SLAKE 영어 train 으로 QLoRA 파인튜닝한다.
- 최적화에는 train 만 쓴다. val 은 매 epoch 평가(손실 + 생성 결과의 closed accuracy / open recall)해서 epoch·학습률 선택에만 쓴다. test 는 읽지 않는다.
- 프롬프트는 기존 평가 경로(llava/eval/model_vqa_med.py, conv_mode=simple)와 같은 형식이다.
  prompt  : system###Human: {질문}\n<im_start><im_patch>*256<im_end>###     (모델이 'Assistant: 답###' 을 생성)
  target  : 'Assistant: {정답}###'
- 실행 위치: /content/LLaVA-Med (llava 패키지가 import 되는 곳), venv_qlora 의 python 으로 실행.
"""
import argparse, json, os, re, glob, time
from collections import Counter

import torch
import transformers
from PIL import Image
from torch.utils.data import Dataset
from transformers import (AutoConfig, AutoTokenizer, BitsAndBytesConfig, CLIPImageProcessor, CLIPVisionModel,
                          Trainer, TrainingArguments, TrainerCallback)
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb
from llava import LlavaLlamaForCausalLM
from llava.conversation import conv_templates

IGNORE = -100
IM_PATCH, IM_START, IM_END = '<im_patch>', '<im_start>', '<im_end>'

ap = argparse.ArgumentParser()
ap.add_argument('--model', required=True)            # 병합된 LLaVA-Med 7B (instruct) 경로
ap.add_argument('--train_json', required=True)
ap.add_argument('--val_json', required=True)
ap.add_argument('--image_folder', required=True)
ap.add_argument('--output_dir', required=True)
ap.add_argument('--lr', type=float, required=True)
ap.add_argument('--epochs', type=float, default=3)
ap.add_argument('--max_steps', type=int, default=-1)  # 샌티 체크용 (>0 이면 epochs 무시)
ap.add_argument('--limit_train', type=int, default=0)
ap.add_argument('--limit_val', type=int, default=0)
ap.add_argument('--bs', type=int, default=4)
ap.add_argument('--grad_accum', type=int, default=4)  # 유효 배치 16
ap.add_argument('--lora_r', type=int, default=64)
ap.add_argument('--lora_alpha', type=int, default=16)
ap.add_argument('--lora_dropout', type=float, default=0.05)
ap.add_argument('--max_len', type=int, default=1024)
ap.add_argument('--gen_bs', type=int, default=8)
ap.add_argument('--gen_new', type=int, default=24)
ap.add_argument('--logging_steps', type=int, default=10)
ap.add_argument('--seed', type=int, default=42)
ap.add_argument('--no_grad_ckpt', action='store_true')   # gradient checkpointing 을 끈다 (빠르지만 메모리를 더 쓴다)
ap.add_argument('--train_projector', action='store_true')   # mm_projector(이미지 특징→언어 모델 임베딩 변환층)도 함께 학습한다
ap.add_argument('--proj_lr', type=float, default=2e-5)      # mm_projector 학습률 (LoRA 는 --lr)
a = ap.parse_args()

os.makedirs(a.output_dir, exist_ok=True)
if os.path.exists(os.path.join(a.output_dir, 'run_info.json')):
    print('이미 끝난 실행입니다:', a.output_dir); raise SystemExit(0)
transformers.set_seed(a.seed)

# ───────── 채점 (기존 노트북과 같은 정의: closed accuracy / open recall, 참고로 open exact match) ─────────
def normalize(text):
    text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()

def closed_accuracy(pred, gt):
    return float(str(gt).strip().lower() in normalize(pred))

def open_recall(pred, gt):
    g, p = Counter(normalize(gt)), Counter(normalize(pred))
    return sum((g & p).values()) / sum(g.values()) if g else 0.0

def open_em(pred, gt):
    return float(normalize(pred) == normalize(gt))

def score(preds, rows):
    cl = [closed_accuracy(p, r['answer']) for p, r in zip(preds, rows) if r['answer_type'] == 'CLOSED']
    op = [open_recall(p, r['answer']) for p, r in zip(preds, rows) if r['answer_type'] == 'OPEN']
    em = [open_em(p, r['answer']) for p, r in zip(preds, rows) if r['answer_type'] == 'OPEN']
    m = lambda x: 100 * sum(x) / len(x) if x else float('nan')
    return {'closed': m(cl), 'open_recall': m(op), 'open_em': m(em), 'overall': m(cl + op), 'n_closed': len(cl), 'n_open': len(op)}

# ───────── 모델 (4비트) ─────────
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
cfg = AutoConfig.from_pretrained(a.model)
vt_name = getattr(cfg, 'mm_vision_tower', 'openai/clip-vit-large-patch14')
if hasattr(cfg, 'mm_vision_tower'):
    delattr(cfg, 'mm_vision_tower')   # 모델 생성자가 CLIP 을 중첩 로드하지 않게 한다 (device_map 로딩과 충돌 방지). 아래에서 직접 붙인다.
if not getattr(cfg, 'use_mm_proj', False):
    cfg.use_mm_proj, cfg.mm_hidden_size = True, 1024
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=dtype, llm_int8_skip_modules=['mm_projector', 'lm_head'])
model = LlavaLlamaForCausalLM.from_pretrained(a.model, config=cfg, quantization_config=bnb_cfg,
                                              torch_dtype=dtype, device_map={'': 0})
tok = AutoTokenizer.from_pretrained(a.model)
processor = CLIPImageProcessor.from_pretrained(vt_name)
vt = CLIPVisionModel.from_pretrained(vt_name, torch_dtype=dtype).cuda()
vt.requires_grad_(False)
model.model.vision_tower = [vt]
model.config.mm_vision_tower = vt_name
vc = vt.config
vc.im_patch_token = tok.convert_tokens_to_ids([IM_PATCH])[0]
vc.use_im_start_end = bool(getattr(model.config, 'mm_use_im_start_end', False))
vc.im_start_token, vc.im_end_token = tok.convert_tokens_to_ids([IM_START, IM_END])
assert vc.use_im_start_end, 'mm_use_im_start_end 가 꺼져 있습니다 (평가 경로와 형식이 달라집니다)'
assert tok.unk_token_id not in (vc.im_patch_token, vc.im_start_token, vc.im_end_token), '이미지 토큰이 토크나이저에 없습니다'
N_PATCH = (vc.image_size // vc.patch_size) ** 2
print('dtype', dtype, '| vision', vt_name, '| image tokens', N_PATCH)

model.config.use_cache = False
model.enable_input_require_grads()      # gradient checkpointing + LoRA: 임베딩 출력이 grad 를 요구하게 한다
if not a.no_grad_ckpt:
    model.gradient_checkpointing_enable()
print('gradient checkpointing:', not a.no_grad_ckpt)
targets = sorted({n.split('.')[-1] for n, m in model.named_modules()
                  if isinstance(m, bnb.nn.Linear4bit) and 'mm_projector' not in n and 'lm_head' not in n})
print('LoRA target modules:', targets)
assert targets, 'Linear4bit 모듈을 찾지 못했습니다 (4비트 로딩 확인)'
model = get_peft_model(model, LoraConfig(r=a.lora_r, lora_alpha=a.lora_alpha, lora_dropout=a.lora_dropout,
                                         target_modules=targets, bias='none', task_type='CAUSAL_LM',
                                         modules_to_save=['mm_projector'] if a.train_projector else None))
if a.train_projector:
    # modules_to_save 는 어댑터 체크포인트에 mm_projector 가중치를 함께 저장·복원하게 해 준다 (best epoch 복원 포함).
    # 작은 학습률에서 bf16 가중치 갱신이 반올림으로 사라지지 않도록 학습 대상 mm_projector 는 fp32 로 올린다 (순전파는 autocast).
    _p = [(n, p) for n, p in model.named_parameters() if 'mm_projector' in n and 'modules_to_save' in n]
    assert _p and all(p.requires_grad for _, p in _p), 'mm_projector 학습 파라미터를 찾지 못했습니다'
    for _, p in _p:
        p.data = p.data.float()
    print('mm_projector 학습:', [(n.split('mm_projector.')[-1], tuple(p.shape), p.dtype) for n, p in _p], '| proj_lr', a.proj_lr)
model.print_trainable_parameters()

# ───────── 데이터 ─────────
IM_IDS = tok.convert_tokens_to_ids([IM_START, IM_PATCH, IM_END])
MARK = '<<IMG>>'

def prompt_ids(r):
    """평가 경로와 같은 프롬프트의 토큰 id. 이 환경(transformers 4.31)의 토크나이저는 문자열 속 '<im_patch>' 를 한 토큰으로 인식하지 못하고
    여러 조각으로 쪼개므로(시퀀스가 ~1100 토큰이 됨), 이미지 토큰 id 는 직접 끼워 넣는다. 앞·뒤 글자만 토크나이저로 변환한다."""
    conv = conv_templates['simple'].copy()
    conv.append_message(conv.roles[0], r['question'] + '\n' + MARK)
    pre, post = conv.get_prompt().split(MARK)
    return tok(pre).input_ids + [IM_IDS[0]] + [IM_IDS[1]] * N_PATCH + [IM_IDS[2]] + tok(post, add_special_tokens=False).input_ids

def load_pixel(r):
    img = Image.open(os.path.join(a.image_folder, r['image'])).convert('RGB')
    return processor.preprocess(img, return_tensors='pt')['pixel_values'][0]

class QADataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        p_ids = prompt_ids(r)
        a_ids = tok('Assistant: ' + r['answer'] + '###', add_special_tokens=False).input_ids
        ids, labels = p_ids + a_ids, [IGNORE] * len(p_ids) + a_ids
        assert len(ids) <= a.max_len, f'시퀀스가 max_len 을 넘습니다: {len(ids)}'
        return {'input_ids': torch.tensor(ids), 'labels': torch.tensor(labels), 'image': load_pixel(r).to(dtype)}

def collate(batch):
    ids = torch.nn.utils.rnn.pad_sequence([b['input_ids'] for b in batch], batch_first=True, padding_value=tok.pad_token_id)
    labels = torch.nn.utils.rnn.pad_sequence([b['labels'] for b in batch], batch_first=True, padding_value=IGNORE)
    return {'input_ids': ids, 'labels': labels, 'attention_mask': ids.ne(tok.pad_token_id),
            'images': torch.stack([b['image'] for b in batch])}

train_rows = json.load(open(a.train_json)); val_rows = json.load(open(a.val_json))
if a.limit_train: train_rows = train_rows[:a.limit_train]
if a.limit_val and a.limit_val < len(val_rows):
    import random
    val_rows = random.Random(a.seed).sample(val_rows, a.limit_val)   # 시드 고정 무작위 부분집합 (모든 학습률이 같은 문항으로 평가)
print(f'train {len(train_rows)} / val {len(val_rows)}')
_ex = prompt_ids(train_rows[0])
print('예시 프롬프트 길이', len(_ex), '| 이미지 토큰 수', sum(t == IM_IDS[1] for t in _ex), '| 앞 8토큰', tok.convert_ids_to_tokens(_ex[:8]))
assert sum(t == IM_IDS[1] for t in _ex) == N_PATCH and len(_ex) < 700, '프롬프트 토큰 구성이 예상과 다릅니다'

# ───────── val 생성 평가 (매 epoch) ─────────
@torch.no_grad()
def generate_val(m):
    m.eval()
    preds = []
    for i in range(0, len(val_rows), a.gen_bs):
        chunk = val_rows[i:i + a.gen_bs]
        seqs = [prompt_ids(r) for r in chunk]
        L = max(len(s) for s in seqs)
        ids = torch.tensor([[tok.pad_token_id] * (L - len(s)) + s for s in seqs]).cuda()          # 왼쪽 패딩
        att = torch.tensor([[0] * (L - len(s)) + [1] * len(s) for s in seqs]).cuda()
        imgs = torch.stack([load_pixel(r) for r in chunk]).to(device='cuda', dtype=dtype)
        with torch.autocast('cuda', dtype=dtype):      # fp32 mm_projector 와 bf16 특징이 섞여도 되게 한다
            out = m.generate(input_ids=ids, attention_mask=att, images=imgs, do_sample=False, max_new_tokens=a.gen_new, use_cache=True)
        for t in tok.batch_decode(out[:, L:], skip_special_tokens=True):
            t = t.split('###')[0].strip()
            preds.append(re.sub(r'^Assistant:\s*', '', t).strip())
    return preds

class ValGen(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, metrics=None, **kw):
        t0 = time.time()
        preds = generate_val(model)
        s = score(preds, val_rows)
        metrics.update({'eval_val_closed': s['closed'], 'eval_val_open_recall': s['open_recall'],
                        'eval_val_open_em': s['open_em'], 'eval_val_overall': s['overall']})   # metric_for_best_model 이 읽는다
        ep = float(state.epoch or 0)
        recent = [h['loss'] for h in state.log_history if 'loss' in h and h.get('epoch', 0) > ep - 1.0]
        rec = {'lr': a.lr, 'epoch': round(ep, 3), 'step': state.global_step,
               'train_loss': sum(recent) / len(recent) if recent else None, 'eval_loss': metrics.get('eval_loss'), **s,
               'gen_sec': round(time.time() - t0)}
        with open(os.path.join(a.output_dir, 'val_metrics.jsonl'), 'a') as f:
            f.write(json.dumps(rec) + '\n')
        json.dump([{'pred': p, 'answer': r['answer'], 'answer_type': r['answer_type']} for p, r in zip(preds, val_rows)],
                  open(os.path.join(a.output_dir, f'val_pred_step{state.global_step}.json'), 'w'))
        print('[VAL]', json.dumps(rec), flush=True)

# ───────── 학습 ─────────
sanity = a.max_steps > 0
strat = dict(evaluation_strategy='steps', eval_steps=a.max_steps, save_strategy='steps', save_steps=a.max_steps) if sanity \
    else dict(evaluation_strategy='epoch', save_strategy='epoch')
args = TrainingArguments(
    output_dir=a.output_dir, per_device_train_batch_size=a.bs, per_device_eval_batch_size=a.bs,
    gradient_accumulation_steps=a.grad_accum, num_train_epochs=a.epochs, max_steps=a.max_steps,
    learning_rate=a.lr, lr_scheduler_type='cosine', warmup_ratio=0.03, weight_decay=0.0, max_grad_norm=0.3,
    bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16), optim='paged_adamw_32bit',
    logging_steps=a.logging_steps, save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model='val_overall', greater_is_better=True, report_to='none',
    remove_unused_columns=False, label_names=['labels'], dataloader_num_workers=2, seed=a.seed, **strat)
class ProjLRTrainer(Trainer):
    """LoRA 와 mm_projector 에 서로 다른 학습률을 주는 옵티마이저 (스케줄러는 각 그룹의 기준 학습률에 같은 비율을 곱한다)."""
    def create_optimizer(self):
        if self.optimizer is None:
            proj = [p for n, p in self.model.named_parameters() if p.requires_grad and 'mm_projector' in n]
            rest = [p for n, p in self.model.named_parameters() if p.requires_grad and 'mm_projector' not in n]
            cls, kw = Trainer.get_optimizer_cls_and_kwargs(self.args)
            kw.pop('lr', None)
            self.optimizer = cls([{'params': rest, 'lr': self.args.learning_rate}, {'params': proj, 'lr': a.proj_lr}], **kw)
        return self.optimizer

trainer = (ProjLRTrainer if a.train_projector else Trainer)(model=model, args=args, train_dataset=QADataset(train_rows), eval_dataset=QADataset(val_rows),
                                                              data_collator=collate, callbacks=[ValGen()])
resume = bool(glob.glob(os.path.join(a.output_dir, 'checkpoint-*')))
print('resume_from_checkpoint =', resume)
trainer.train(resume_from_checkpoint=resume or None)

best = trainer.state.best_model_checkpoint
ckpts = sorted(glob.glob(os.path.join(a.output_dir, 'checkpoint-*')), key=lambda d: int(d.rsplit('-', 1)[1]))
info = {'lr': a.lr, 'epochs': a.epochs, 'max_steps': a.max_steps, 'best_checkpoint': best, 'last_checkpoint': ckpts[-1] if ckpts else None,
        'best_val_overall': trainer.state.best_metric, 'n_train': len(train_rows), 'n_val': len(val_rows),
        'train_projector': a.train_projector, 'proj_lr': a.proj_lr if a.train_projector else None,
        'grad_ckpt': not a.no_grad_ckpt, 'peak_gpu_mem_gb': round(torch.cuda.max_memory_allocated() / 2**30, 1)}
json.dump(trainer.state.log_history, open(os.path.join(a.output_dir, 'log_history.json'), 'w'))
json.dump(info, open(os.path.join(a.output_dir, 'run_info.json'), 'w'), indent=1)
print('DONE', json.dumps(info))


In [ ]:
%%writefile /content/LLaVA-Med/merge_qlora.py
"""QLoRA 어댑터를 fp16 LLaVA-Med 위에 병합해 기존 추론 경로(model_vqa_med.py)가 읽는 HF 체크포인트로 저장한다.
참고: 어댑터는 4비트 양자화된 기반 위에서 학습했고 병합은 fp16 기반 위에서 하므로 양자화 오차만큼 출력이 미세하게 달라질 수 있다.
"""
import argparse, os, torch
from transformers import AutoConfig, AutoTokenizer
from peft import PeftModel
from llava import LlavaLlamaForCausalLM

ap = argparse.ArgumentParser()
ap.add_argument('--base', required=True)      # 병합된 LLaVA-Med 7B (instruct)
ap.add_argument('--adapter', required=True)   # Trainer 체크포인트 폴더 (adapter_model.bin 이 있는 곳)
ap.add_argument('--out', required=True)
a = ap.parse_args()

assert os.path.exists(os.path.join(a.adapter, 'adapter_config.json')), f'어댑터가 없습니다: {a.adapter}'
cfg = AutoConfig.from_pretrained(a.base)
vt_name = getattr(cfg, 'mm_vision_tower', 'openai/clip-vit-large-patch14')
if hasattr(cfg, 'mm_vision_tower'):
    delattr(cfg, 'mm_vision_tower')
model = LlavaLlamaForCausalLM.from_pretrained(a.base, config=cfg, torch_dtype=torch.float16, device_map={'': 0})
model = PeftModel.from_pretrained(model, a.adapter).merge_and_unload()

# mm_projector 를 함께 학습한 어댑터라면(modules_to_save) 그 가중치를 확실히 반영한다 (peft 의 병합 동작에 기대지 않고 직접 읽어서 넣는다)
sd = None
for fn in ('adapter_model.bin', 'adapter_model.safetensors'):
    pth = os.path.join(a.adapter, fn)
    if os.path.exists(pth):
        if fn.endswith('.bin'):
            sd = torch.load(pth, map_location='cpu')
        else:
            from safetensors.torch import load_file
            sd = load_file(pth)
        break
proj = {k.split('mm_projector.')[-1].replace('modules_to_save.default.', ''): v for k, v in (sd or {}).items()
        if 'mm_projector' in k}      # peft 버전에 따라 키가 '...mm_projector.weight' 또는 '...mm_projector.modules_to_save.default.weight' 로 저장된다 (둘 다 읽는다)
if proj:
    old = model.model.mm_projector
    new = torch.nn.Linear(proj['weight'].shape[1], proj['weight'].shape[0])
    new.load_state_dict({k: v.float() for k, v in proj.items()})
    new = new.half().to(next(model.parameters()).device)
    print('mm_projector 반영: 병합 전과의 최대 차이', float((new.weight - old.weight.to(new.weight.dtype)).abs().max()) if hasattr(old, 'weight') else '(래퍼)')
    model.model.mm_projector = new
else:
    print('어댑터에 mm_projector 가중치가 없습니다 (LoRA 만 학습한 어댑터)')
model.config.mm_vision_tower = vt_name
model.config.use_cache = True
os.makedirs(a.out, exist_ok=True)
model.save_pretrained(a.out, safe_serialization=False, max_shard_size='5GB')
# 토크나이저는 다시 저장하지 않고 기반 폴더의 파일을 그대로 복사한다 (이 환경의 transformers 4.31 로 다시 저장하면 추가 토큰 정보가 달라질 수 있다)
import shutil
for f in ('tokenizer.model', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json'):
    if os.path.exists(os.path.join(a.base, f)):
        shutil.copy(os.path.join(a.base, f), os.path.join(a.out, f))
print('merged ->', a.out)


In [ ]:
# C-4. 설정 (학습률만 다르게, 나머지는 train_qlora_slake.py 의 기본값으로 동일)
import os, json, subprocess, glob, math
QROOT = f'{DRIVE_ROOT}/qlora_slake'
LRS = [2e-5, 1e-4, 2e-4]
EPOCHS = 2          # 1단계: 학습률 3개를 비교하는 학습 길이 (비용 절감: 학습률마다 이 길이로만 돈다)
VAL_LIMIT_SEARCH = 400   # 1단계에서 epoch 마다 생성 평가할 val 문항 수 (시드 고정 무작위 부분집합, 0=전체). 2단계는 항상 val 전체
FINAL_EPOCHS = 9    # 2단계: val 로 선택된 학습률 하나만 이 길이로 다시 학습 (공식 SLAKE delta 와 같은 9 epoch). 생략해도 된다 (C-9c/9d)
FINAL_TRAINVAL = False    # 2단계를 train+val 로 학습 (규칙 일부 완화 — 결과표에 별도 표기). val 선택이 없으므로 마지막 epoch 를 보고한다. False 면 train 만 학습하고 val 로 선택
FINAL_VAL_LIMIT = 100    # FINAL_TRAINVAL 일 때 epoch 마다 생성 평가할 val 문항 수 (val 이 학습에 들어가 진단용일 뿐이므로 적게)

TRAIN_PROJECTOR = True   # 2단계에서 mm_projector(이미지 특징→언어 모델 변환층)도 LoRA 와 함께 학습 (1단계 학습률 탐색은 LoRA 만 학습했다)
PROJ_LR = 2e-5           # mm_projector 학습률 (LoRA 는 1단계에서 val 로 고른 값). LLaVA 의 LoRA 학습 방식과 같은 값
FINAL_STAGE = 'final_proj' if TRAIN_PROJECTOR else 'final'   # 설정이 다르면 2단계 결과 폴더도 분리
if FINAL_TRAINVAL:
    FINAL_STAGE += '_trainval'
GRAD_CKPT = False   # False: gradient checkpointing 끔 (빠름, 메모리 더 씀). GPU 메모리 부족(OOM)이면 True 로
SANITY_STAGE = 'sanity_gc' if GRAD_CKPT else 'sanity_nogc'   # 설정이 다르면 샌티 체크 결과 폴더도 분리

def lr_dir(stage, lr):
    return f'{QROOT}/{stage}/lr{lr:g}'

def train_cmd(lr, stage, extra=''):
    proj = f'--train_projector --proj_lr {PROJ_LR}' if (TRAIN_PROJECTOR and stage == FINAL_STAGE) else ''
    tv = FINAL_TRAINVAL and stage == FINAL_STAGE
    tj = f'{QDATA}/slake_trainval.json' if tv else f'{QDATA}/slake_train.json'
    if tv:
        extra += f' --limit_val {FINAL_VAL_LIMIT}'
    return (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {QPY} train_qlora_slake.py --model {INSTRUCT_MODEL} '
            f'--train_json {tj} --val_json {QDATA}/slake_val.json --image_folder {SLAKE_IMG_DIR} '
            f'--output_dir {lr_dir(stage, lr)} --lr {lr} {"" if GRAD_CKPT else "--no_grad_ckpt"} {proj} {extra}')
print({lr: lr_dir('full', lr) for lr in LRS})

In [ ]:
# C-7. 1단계(학습률 탐색): 학습률 3개를 차례로 EPOCHS epoch 씩 (백그라운드). 끊겨도 이 셀을 다시 실행하면 체크포인트에서 이어 가고, 끝난 실행은 건너뛴다.
if 'full_proc' in globals() and full_proc.poll() is None:
    print('이미 실행 중입니다. PID', full_proc.pid)
else:
    for lr in LRS:
        os.makedirs(lr_dir('full', lr), exist_ok=True)
    cmds = ' ; '.join(train_cmd(lr, 'full', f'--epochs {EPOCHS} --limit_val {VAL_LIMIT_SEARCH}') + f' >> {lr_dir("full", lr)}/stdout.log 2>&1' for lr in LRS)
    full_proc = subprocess.Popen(['bash', '-c', cmds], start_new_session=True)
    print('PID', full_proc.pid)

In [ ]:
# C-8. 진행 확인 (epoch 마다 val 손실과 val 생성 점수가 쌓인다)
print('프로세스 종료 코드(None=실행 중):', full_proc.poll())
for lr in LRS:
    p = f'{lr_dir("full", lr)}/val_metrics.jsonl'
    print(f'--- lr={lr:g}')
    if os.path.exists(p):
        for l in open(p):
            v = json.loads(l)
            print(f'  epoch {v["epoch"]:.2f}  train_loss {v["train_loss"]:.4f}  val_loss {v["eval_loss"]:.4f}  closed {v["closed"]:.2f}  open recall {v["open_recall"]:.2f}  open EM {v["open_em"]:.2f}  overall {v["overall"]:.2f}')
    print('  로그 끝:', ''.join(open(f'{lr_dir("full", lr)}/stdout.log', errors='ignore').readlines()[-2:]).strip()[:200] if os.path.exists(f'{lr_dir("full", lr)}/stdout.log') else '(없음)')

In [ ]:
# C-9. 선택: train 손실이 아니라 val 성능(overall, 동점이면 val 손실)으로 학습률·epoch 를 고른다. test 는 쓰지 않는다.
import numpy as np
import matplotlib.pyplot as plt

runs = {}
for lr in LRS:
    p = f'{lr_dir("full", lr)}/val_metrics.jsonl'
    if os.path.exists(p):
        runs[lr] = [json.loads(l) for l in open(p)]
assert runs, 'val_metrics.jsonl 이 없습니다 (본 실험 결과 필요)'

def best_of(recs):
    return max(recs, key=lambda v: (v['overall'], -v['eval_loss']))

print(f"{'LR':>8s} {'Epochs':>6s} {'Val-selected epoch':>19s} {'Val open':>9s} {'Val closed':>10s} {'Val overall':>11s} {'Train loss':>10s} {'Val loss':>9s} {'Last-epoch val overall':>23s}")
sel_lr, sel_rec = None, None
for lr, recs in runs.items():
    b, last = best_of(recs), recs[-1]
    print(f"{lr:8g} {int(round(last['epoch'])):6d} {int(round(b['epoch'])):19d} {b['open_recall']:9.2f} {b['closed']:10.2f} {b['overall']:11.2f} {b['train_loss']:10.4f} {b['eval_loss']:9.4f} {last['overall']:23.2f}")
    if sel_rec is None or (b['overall'], -b['eval_loss']) > (sel_rec['overall'], -sel_rec['eval_loss']):
        sel_lr, sel_rec = lr, b
info = json.load(open(f'{lr_dir("full", sel_lr)}/run_info.json'))
SEL = {'lr': sel_lr, 'epoch': sel_rec['epoch'], 'best_checkpoint': info['best_checkpoint'], 'last_checkpoint': info['last_checkpoint'], 'val': sel_rec}
print(f"\n선택: lr={sel_lr:g}, epoch {sel_rec['epoch']:.0f} (val overall {sel_rec['overall']:.2f})  ← train 손실이 아니라 val 성능 기준")
print('best ckpt:', SEL['best_checkpoint'], '| last ckpt:', SEL['last_checkpoint'])

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
for lr, recs in runs.items():
    ep = [v['epoch'] for v in recs]
    ax[0, 0].plot(ep, [v['train_loss'] for v in recs], 'o-', label=f'{lr:g}')
    ax[0, 1].plot(ep, [v['eval_loss'] for v in recs], 'o-', label=f'{lr:g}')
    ax[1, 0].plot(ep, [v['overall'] for v in recs], 'o-', label=f'{lr:g}')
    l, = ax[1, 1].plot(ep, [v['closed'] for v in recs], 'o-', label=f'closed {lr:g}')
    ax[1, 1].plot(ep, [v['open_recall'] for v in recs], 's--', color=l.get_color(), label=f'open recall {lr:g}')
for a_, t in zip(ax.ravel(), ['(a) train loss', '(b) val loss', '(c) val overall per epoch', '(d) val closed (solid) / open recall (dashed)']):
    a_.set_title(t); a_.set_xlabel('epoch'); a_.legend(fontsize=7)
ax[0, 0].set_yscale('log')
fig.suptitle('SLAKE QLoRA (LLaVA-Med v1.0) — learning-rate comparison, selected on validation only')
plt.tight_layout(); plt.savefig(f'{QROOT}/val_curves.png', dpi=150); plt.show()

In [ ]:
# C-9c. (선택) 2단계: C-9 에서 val 로 선택된 학습률 하나만 FINAL_EPOCHS epoch 로 길게 학습 (백그라운드). 생략하면 C-9 의 선택(짧은 학습)을 그대로 쓴다.
assert 'SEL' in globals(), '먼저 C-9 로 학습률을 선택하세요'
if 'final_proc' in globals() and final_proc.poll() is None:
    print('이미 실행 중입니다. PID', final_proc.pid)
else:
    SEARCH_SEL = SEL                      # 1단계 선택 결과 보관
    d = lr_dir(FINAL_STAGE, SEL['lr']); os.makedirs(d, exist_ok=True)
    cmd = train_cmd(SEL['lr'], FINAL_STAGE, f'--epochs {FINAL_EPOCHS}') + f' >> {d}/stdout.log 2>&1'
    final_proc = subprocess.Popen(['bash', '-c', cmd], start_new_session=True)
    print(f'lr={SEL["lr"]:g} 로 {FINAL_EPOCHS} epoch 학습 시작. PID', final_proc.pid, '| 진행은 C-8 과 같은 방식으로 아래 로그/val_metrics 를 확인:', d)

In [ ]:
# C-9d. 2단계 결과로 선택 갱신: 긴 학습의 val 결과에서 epoch 를 다시 고른다 (train 손실이 아니라 val 성능). 이후 C-9b/C-10 이 이 선택을 쓴다.
lr_ = SEARCH_SEL['lr'] if 'SEARCH_SEL' in globals() else SEL['lr']
d = lr_dir(FINAL_STAGE, lr_)
print('프로세스 종료 코드(None=실행 중):', final_proc.poll() if 'final_proc' in globals() else '(미실행)')
vm = f'{d}/val_metrics.jsonl'
if not os.path.exists(vm):
    lg = f'{d}/stdout.log'
    tail = ''.join(open(lg, errors='ignore').readlines()[-12:]) if os.path.exists(lg) else '(로그 파일 없음 → C-9c 를 아직 실행하지 않았거나 시작 전에 실패)'
    raise AssertionError('아직 val 결과가 없습니다. 첫 epoch 가 끝나야 생깁니다(약 30분). 학습 프로세스가 죽었는지 아래 로그 끝을 확인하세요.\n--- ' + lg + ' 끝 ---\n' + tail)
recs = [json.loads(l) for l in open(vm)]
for v in recs:
    print(f'  epoch {v["epoch"]:.2f}  train_loss {v["train_loss"]:.4f}  val_loss {v["eval_loss"]:.4f}  closed {v["closed"]:.2f}  open recall {v["open_recall"]:.2f}  overall {v["overall"]:.2f}')
assert os.path.exists(f'{d}/run_info.json'), '긴 학습이 아직 끝나지 않았습니다 (run_info.json 없음)'
info = json.load(open(f'{d}/run_info.json'))
b = best_of(recs)
if FINAL_TRAINVAL:      # train+val 학습: val 선택이 의미 없으므로 마지막 epoch 를 사용한다
    b = recs[-1]; info['best_checkpoint'] = info['last_checkpoint']
    print('※ train+val 학습(규칙 일부 완화): val 선택 없이 마지막 epoch 사용 — 결과표에 별도 표기하세요')
runs_search, runs = runs, {lr_: recs}
SEL = {'lr': lr_, 'epoch': b['epoch'], 'best_checkpoint': info['best_checkpoint'], 'last_checkpoint': info['last_checkpoint'], 'val': b, 'dir': d}
print(f"\n갱신된 선택: lr={lr_:g}, epoch {b['epoch']:.0f} / {recs[-1]['epoch']:.0f} (val overall {b['overall']:.2f}, closed {b['closed']:.2f}, open recall {b['open_recall']:.2f})")

In [ ]:
# C-9b. test 문항 파일 준비 + 겹침 점검 — 여기서 처음 test 를 읽는다. C-9 에서 학습률·epoch 선택이 끝난 뒤에만 실행한다.
assert 'SEL' in globals(), '먼저 C-9 로 학습률·epoch 를 선택하세요'
slake_test = [r for r in json.load(open(hf_hub_download('BoKelvin/SLAKE', 'test.json', repo_type='dataset'))) if r['q_lang'] == 'en']
assert len(slake_test) == 1061, len(slake_test)
for r in slake_test:
    r['answer_type'] = r['answer_type'].upper()
write_eval_files('slake', slake_test, lambda r: r['img_name'])
assert all(os.path.exists(os.path.join(SLAKE_IMG_DIR, r['img_name'])) for r in slake_test), '이미지 누락'

# 분할 간 겹침 (이미지+질문): 학습 데이터와 test 가 겹치면 결과 해석에 주의
_trn, _val = json.load(open(f'{QDATA}/slake_train.json')), json.load(open(f'{QDATA}/slake_val.json'))
_k = lambda img, q: (img, q)
_te = {_k(r['img_name'], r['question'] + (CLOSED_SUFFIX if r['answer_type'] == 'CLOSED' else '')) for r in slake_test}
print('train∩test:', len({_k(r['image'], r['question']) for r in _trn} & _te), '| val∩test:', len({_k(r['image'], r['question']) for r in _val} & _te))

In [ ]:
# C-10. 선택된 설정에 대해서만 test 추론 (여기서 처음 test 를 읽는다). 어댑터 병합 → 기존 추론 경로로 평가. 병합 모델은 약 13GB 라 끝나면 지워도 된다.
assert 'SEL' in globals(), '먼저 C-9 로 학습률·epoch 를 선택하세요'
assert 'do_sample=False' in open('/content/LLaVA-Med/llava/eval/model_vqa_med.py').read(), 'greedy 패치(앞 절)가 필요합니다'
TEST_QFILE = f'{EVAL_DIR}/slake_test.json'
assert os.path.exists(TEST_QFILE), '먼저 C-9b 로 test 문항 파일을 만드세요'

def merge_and_infer(tag, ckpt):
    out = f'/content/llava-med-7b-slake-qlora-{tag}'
    sh(f'cd /content/LLaVA-Med && {QPY} merge_qlora.py --base {INSTRUCT_MODEL} --adapter {ckpt} --out {out}')
    cmd = (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_med.py --model-name {out} '
           f'--question-file {TEST_QFILE} --image-folder {SLAKE_IMG_DIR} --answers-file {DRIVE_ROOT}/slake_qlora_{tag}_pred.jsonl '
           f'> /content/slake_qlora_{tag}.log 2>&1')
    return subprocess.Popen(['bash', '-c', cmd], start_new_session=True)

test_procs = {'best': merge_and_infer('best', SEL['best_checkpoint'])}
print('best(val 선택 epoch) 추론 PID', test_procs['best'].pid)

In [ ]:
# C-11. (선택) 마지막 epoch 체크포인트도 같은 방식으로 추론 — 보고표의 "Last-epoch overall" 용. best 가 마지막 epoch 와 같으면 건너뛴다.
if SEL['last_checkpoint'] == SEL['best_checkpoint']:
    print('best 가 마지막 epoch 와 같습니다 → 같은 결과를 재사용')
else:
    assert test_procs['best'].poll() is not None, '먼저 best 추론이 끝나야 합니다 (디스크/GPU 점유)'
    test_procs['last'] = merge_and_infer('last', SEL['last_checkpoint'])
    print('last 추론 PID', test_procs['last'].pid)

In [ ]:
# C-12. 진행 확인
for tag, pr in test_procs.items():
    p = f'{DRIVE_ROOT}/slake_qlora_{tag}_pred.jsonl'
    n = sum(1 for _ in open(p)) if os.path.exists(p) else 0
    print(f'{tag}: {n}/1061 | 종료 코드(None=실행 중): {pr.poll()}')

In [ ]:
# C-13. 최종 보고표 (기존 채점: closed accuracy / open recall). Best epoch 는 val 로 고른 값이다.
if FINAL_TRAINVAL:
    print('※ SLAKE QLoRA: train+val 학습 + 9 epoch 마지막 epoch (규칙 일부 완화). 규칙 준수 결과(train 만 학습, val 선택, 6 epoch: 79.40)와 별도 행으로 표기하세요.\n')
paper_slake = {'open': 83.08, 'closed': 85.34, 'overall': 83.97}
res = {}
for tag in ('best', 'last'):
    p = f'{DRIVE_ROOT}/slake_qlora_{tag}_pred.jsonl'
    if tag == 'last' and SEL['last_checkpoint'] == SEL['best_checkpoint']:
        res['last'] = res.get('best'); continue
    if os.path.exists(p) and sum(1 for _ in open(p)) == 1061:
        res[tag] = score_jsonl(f'{EVAL_DIR}/slake_ref.json', p)
m, ml = res.get('best'), res.get('last')
assert m is not None, 'best 추론이 아직 끝나지 않았습니다'
epochs = int(round(runs[SEL['lr']][-1]['epoch']))
print(f"{'Dataset':8s} {'LR':>8s} {'Epochs':>6s} {'Best epoch(val)':>15s} {'Open':>7s} {'Closed':>7s} {'Overall':>8s} {'Paper overall':>13s} {'Delta':>7s} {'Last-epoch overall':>19s}")
print(f"{'SLAKE':8s} {SEL['lr']:8g} {epochs:6d} {int(round(SEL['epoch'])):15d} {m['open']:7.2f} {m['closed']:7.2f} {m['overall']:8.2f} {paper_slake['overall']:13.2f} {m['overall'] - paper_slake['overall']:+7.2f} "
      + (f"{ml['overall']:19.2f}" if ml else f"{'(미실행)':>19s}"))
print('※ QLoRA 근사: 논문 LLaVA-Med(전체 파인튜닝)와 조건이 다르므로 Delta 는 재현 오차가 아니라 방식 차이를 포함합니다. 학습률·epoch 는 val 로만 선택, test 는 위 추론에서 처음 사용.')
json.dump({'selected': SEL, 'test_best': m, 'test_last': ml}, open(f'{QROOT}/final_report.json', 'w'), indent=1, default=str)

In [ ]:
%%writefile /content/curve_plot.py
"""모델 공통 학습 곡선 그림 (2x2). 선택 기준은 항상 val 이고, test 곡선은 진단용으로만 겹쳐 그린다.

  (a) train loss (선택: 학습률)
  (b) val overall vs test overall, 세로선 = val 로 고른 epoch
  (c) epoch 별 open / closed / overall  (실선 = val, 점선 = test, 회색 가로 점선 = 논문 test 값)
  (d) "이 epoch 에서 학습을 멈췄다면" 보고했을 값: 그 시점까지 val overall 이 가장 높았던 epoch 의 test 값 (test 가 있는 epoch 만).
      test 가 없으면 val 의 best-so-far 를 그린다. 어느 쪽이든 선택에 test 는 쓰지 않는다.

입력 형식 (epoch 는 0 부터 센 정수 또는 1 부터 센 정수 중 하나로 통일해서 넣을 것)
  train_loss : {epoch: float}
  lr         : {epoch: float}                (선택)
  val, test  : {epoch: {'open': %, 'closed': %, 'overall': %}}   test 는 평가한 epoch 만 있어도 된다
  paper      : {'open': %, 'closed': %, 'overall': %}            (선택)
"""
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

KEYS = ('open', 'closed', 'overall')
COL = {'open': 'tab:blue', 'closed': 'tab:orange', 'overall': 'tab:green'}


def select_by_val(val):
    """val overall 이 최대인 epoch (동점이면 이른 epoch)."""
    return max(sorted(val), key=lambda e: (val[e]['overall'], -e))


def best_so_far(val):
    """각 epoch 까지의 val-best epoch 를 {epoch: 그때의 best epoch} 로 돌려준다."""
    out, best = {}, None
    for e in sorted(val):
        if best is None or val[e]['overall'] > val[best]['overall']:
            best = e
        out[e] = best
    return out


def _sparse(test, val):
    """test 가 5개 미만(보통 val 선택 epoch 와 마지막 epoch 두 점)이면 곡선이 아니므로 report 레이아웃(막대 비교)을 쓴다.
    5 epoch 마다 평가하는 등 5개 이상이면 점을 선으로 이은 곡선 레이아웃을 쓴다."""
    return len(test) < 5


def _bars(a, groups, series, ylim=None):
    """groups: 막대 묶음 이름 목록, series: [(label, [값...], 스타일 dict)]"""
    n = len(series); w = 0.8 / n
    for j, (lab, vals, kw) in enumerate(series):
        xs = [i + (j - (n - 1) / 2) * w for i in range(len(groups))]
        a.bar(xs, [0 if v is None else v for v in vals], w * 0.95, label=lab, **kw)
        for x, v in zip(xs, vals):
            if v is not None:
                a.text(x, v + 0.8, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
    a.set_xticks(range(len(groups))); a.set_xticklabels(groups); a.set_ylabel('%')
    if ylim:
        a.set_ylim(*ylim)


def _ticks(axes, val, test, train_loss):
    last = max([max(val)] + ([max(test)] if test else []) + ([max(train_loss)] if train_loss else []))
    step = 5 if last >= 15 else 1
    for a in axes:
        a.xaxis.set_major_locator(MultipleLocator(step)); a.xaxis.set_major_formatter(FormatStrFormatter('%d')); a.grid(axis='x', alpha=0.2)


def _plot_report(title, train_loss, lr, val, test, paper, coverage, out_png, sel):
    """test 를 한두 번만 평가한 경우의 레이아웃: 선택 근거(val) + 최종 test 비교 + 후보 coverage."""
    fig, ax = plt.subplots(2, 2, figsize=(14, 8.5))
    # (a) train loss
    a = ax[0, 0]
    if train_loss:
        xs = sorted(train_loss); a.plot(xs, [train_loss[e] for e in xs], marker='.', label='train loss'); a.set_yscale('log'); a.legend(loc='upper right')
    else:
        a.text(0.5, 0.5, 'no train-loss log found', ha='center', va='center', transform=a.transAxes, color='gray')
    a.set_title('(a) train loss'); a.set_xlabel('epoch')
    # (b) val 곡선 = 선택 근거
    a = ax[0, 1]
    for k in KEYS:
        xs = sorted(val); a.plot(xs, [val[e][k] for e in xs], color=COL[k], lw=1.6 if k == 'overall' else 1.0, label=f'val {k}')
    a.axvline(sel, color='green', lw=1.0, label=f'val-selected epoch {sel}')
    a.set_title('(b) validation per epoch (the only signal used for selection)', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right', fontsize=8)
    # (c) test 비교
    a = ax[1, 0]
    series = []
    if sel in test:
        series.append((f'val-selected (epoch {sel})', [test[sel][k] for k in KEYS], dict(color='tab:blue')))
    last = max(test) if test else None
    if last is not None and last != sel:
        series.append((f'last epoch ({last})', [test[last][k] for k in KEYS], dict(color='tab:cyan')))
    if paper:
        series.append(('paper', [paper.get(k) for k in KEYS], dict(color='lightgray', hatch='//', edgecolor='gray')))
    if series:
        _bars(a, KEYS, series, ylim=(0, 120)); a.legend(loc='upper center', fontsize=8, ncol=3)
    a.set_title('(c) test score, evaluated once per checkpoint (never used for selection)', fontsize=10)
    # (d) candidate coverage
    a = ax[1, 1]
    if coverage:
        cov = [coverage[k]['coverage'] if isinstance(coverage[k], dict) else coverage[k] for k in KEYS]
        _bars(a, KEYS, [('candidate coverage (answer is in the train-only list)', cov, dict(color='lightsteelblue'))], ylim=(0, 120))
        if sel in test:
            a.plot(range(len(KEYS)), [test[sel][k] for k in KEYS], ls='none', marker='D', ms=8, color='tab:blue', markeredgecolor='k', label=f'test score (epoch {sel})')
        a.set_title('(d) candidate coverage of test answers vs achieved score', fontsize=10); a.legend(loc='upper center', fontsize=8, ncol=2)
    else:
        bs = best_so_far(val); xs = sorted(bs)
        for k in KEYS:
            a.step(xs, [val[bs[e]][k] for e in xs], where='post', color=COL[k], label=f'val {k}')
        a.axvline(sel, color='green', lw=1.0)
        a.set_title('(d) val best-so-far (selection trace)', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right', fontsize=8)
    if sel in test:
        t = test[sel]; head = f"val-selected epoch {sel}: test overall {t['overall']:.2f} / open {t['open']:.2f} / closed {t['closed']:.2f}"
    else:
        v = val[sel]; head = f"val-selected epoch {sel}: val overall {v['overall']:.2f} / open {v['open']:.2f} / closed {v['closed']:.2f} (test not evaluated)"
    fig.suptitle(f'{title}  |  {head}', fontsize=11)
    _ticks([ax[0, 0], ax[0, 1]] + ([] if coverage else [ax[1, 1]]), val, test, train_loss)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, sel


def closed_confusion(y_true, y_pred):
    """yes/no 문항의 혼동행렬과 클래스별 precision / recall / F1. y_pred 가 yes/no 가 아니면 'other' 로 센다.
    입력은 소문자·구두점 정리된 문자열 목록. 점수(확률)가 아니라 예측 답만 쓴다."""
    cls = ('yes', 'no'); cols = ('yes', 'no', 'other')
    cm = {t: {c: 0 for c in cols} for t in cls}
    for t, p in zip(y_true, y_pred):
        if t in cm:
            cm[t][p if p in ('yes', 'no') else 'other'] += 1
    m = {}
    for c in cls:
        tp = cm[c][c]; pred = sum(cm[t][c] for t in cls); true = sum(cm[c].values())
        pr = tp / pred if pred else 0.0; rc = tp / true if true else 0.0
        m[c] = {'precision': 100 * pr, 'recall': 100 * rc, 'f1': 100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 'support': true}
    m['macro'] = {k: sum(m[c][k] for c in cls) / 2 for k in ('precision', 'recall', 'f1')}
    n = sum(sum(r.values()) for r in cm.values())
    m['accuracy'] = 100 * sum(cm[c][c] for c in cls) / n if n else 0.0
    return cm, m


def plot_closed_f1(title, y_true, y_pred, out_png=None):
    """closed(yes/no) 예측의 혼동행렬 + 클래스별 precision/recall/F1 막대. (ROC 는 확률 점수가 있어야 해서 여기서는 그리지 않는다)"""
    cm, m = closed_confusion(y_true, y_pred)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    cols = ('yes', 'no', 'other'); rows = ('yes', 'no')
    a = ax[0]; a.imshow([[cm[r][c] for c in cols] for r in rows], cmap='Blues')
    for i, r in enumerate(rows):
        for j, c in enumerate(cols):
            a.text(j, i, cm[r][c], ha='center', va='center', fontsize=14, color='black')
    a.set_xticks(range(3)); a.set_xticklabels([f'pred {c}' for c in cols]); a.set_yticks(range(2)); a.set_yticklabels([f'true {r}' for r in rows])
    a.set_title(f'confusion matrix (closed, n={sum(sum(v.values()) for v in cm.values())})')
    groups = ('yes', 'no', 'macro')
    _bars(ax[1], groups, [(k, [m[g][k] for g in groups], dict(color=c)) for k, c in (('precision', 'tab:blue'), ('recall', 'tab:orange'), ('f1', 'tab:green'))], ylim=(0, 120))
    ax[1].legend(loc='upper center', fontsize=8, ncol=3); ax[1].set_title(f"precision / recall / F1  (accuracy {m['accuracy']:.1f}%)")
    fig.suptitle(title, fontsize=11); fig.tight_layout(rect=(0, 0, 1, 0.94))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, m


def plot_curves(title, train_loss=None, lr=None, val=None, test=None, paper=None, out_png=None, selected=None, coverage=None):
    val, test = val or {}, test or {}
    sparse = _sparse(test, val)
    if sparse:      # test 가 한두 번만 평가된 경우: 우리 방식(val 선택, test 한 번)에 맞는 레이아웃
        assert val, 'val 곡선이 없습니다 (선택 기준이 필요)'
        return _plot_report(title, train_loss, lr, val, test, paper, coverage, out_png, select_by_val(val) if selected is None else selected)
    tkw = dict(ls='none', marker='o', ms=7) if sparse else dict(marker='o', ms=4)
    assert val, 'val 곡선이 없습니다 (선택 기준이 필요)'
    sel = select_by_val(val) if selected is None else selected
    fig, ax = plt.subplots(2, 2, figsize=(14, 8.5))

    # (a) loss
    a = ax[0, 0]
    if train_loss:
        xs = sorted(train_loss)
        a.plot(xs, [train_loss[e] for e in xs], marker='.', label='train loss')
        a.set_yscale('log')
    if not train_loss:
        a.text(0.5, 0.5, 'no train-loss log found', ha='center', va='center', transform=a.transAxes, color='gray')
    a.set_title('(a) train loss' + (' / learning rate' if lr else '')); a.set_xlabel('epoch')
    h, l = a.get_legend_handles_labels()
    if lr:
        b = a.twinx(); xs = sorted(lr)
        b.plot(xs, [lr[e] for e in xs], ':', color='gray', label='lr'); b.set_ylabel('lr')
        h2, l2 = b.get_legend_handles_labels(); h += h2; l += l2
    if h:
        a.legend(h, l, loc='upper right')

    # (b) val vs test overall
    a = ax[0, 1]
    xs = sorted(val); a.plot(xs, [val[e]['overall'] for e in xs], marker='.', color='tab:orange', label='validation overall')
    if test:
        xt = sorted(test); a.plot(xt, [test[e]['overall'] for e in xt], color='tab:blue', label='test overall (diagnostic)', **tkw)
    a.axvline(sel, color='green', lw=0.8, label=f'val-selected epoch {sel}')
    a.set_title('(b) validation vs test overall'); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right')

    # (c) per-epoch metrics
    a = ax[1, 0]
    for k in KEYS:
        xs = sorted(val); a.plot(xs, [val[e][k] for e in xs], color=COL[k], lw=1.2, label=f'val {k}')
        if test:
            xt = sorted(test); a.plot(xt, [test[e][k] for e in xt], color=COL[k], label=f'test {k}', **({'ls': 'none', 'marker': 'o', 'ms': 7} if sparse else {'ls': '--', 'marker': 'o', 'ms': 3, 'lw': 0.9}))
        if paper and paper.get(k) is not None:
            a.axhline(paper[k], color=COL[k], ls=':', lw=0.9, alpha=0.7)
    a.set_title('(c) accuracy per epoch (solid = val, dashed = test, dotted = paper test)'); a.set_xlabel('epoch'); a.set_ylabel('%')
    a.legend(loc='lower right', fontsize=7, ncol=2)

    # (d) reported value if training stopped at this epoch (val-based selection)
    a = ax[1, 1]; bs = best_so_far(val)
    dense = len(test) >= max(5, 0.5 * len(val))      # test 가 epoch 의 절반 이상이어야 '그 시점 val-best 의 test' 계단을 그릴 수 있다
    if test and dense and any(bs[e] in test for e in bs):
        lab = 'test @ val-best-so-far epoch'
        for k in KEYS:
            pts = [(e, test[bs[e]][k]) for e in sorted(bs) if bs[e] in test]
            a.step([p[0] for p in pts], [p[1] for p in pts], where='post', color=COL[k], label=f'{k} (test)')
    else:
        lab = 'lines: val best-so-far | markers: test at evaluated epochs' if test else 'val best-so-far (no per-epoch test)'
        for k in KEYS:
            a.step(sorted(bs), [val[bs[e]][k] for e in sorted(bs)], where='post', color=COL[k], label=f'{k} (val)')
            if test:
                a.plot(sorted(test), [test[e][k] for e in sorted(test)], ls='none', marker='*', ms=11, color=COL[k], markeredgecolor='k', label=f'{k} (test)')
    if paper:
        for k in KEYS:
            if paper.get(k) is not None:
                a.axhline(paper[k], color='gray', ls=':', lw=0.7)
    a.axvline(sel, color='red', lw=0.8)
    a.set_title('(d) reported value if training stopped at this epoch\n[' + lab + ']', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right')

    if sel in test:
        t = test[sel]; head = f"val-selected epoch {sel}: test overall {t['overall']:.2f} / open {t['open']:.2f} / closed {t['closed']:.2f}"
    else:
        v = val[sel]; head = f"val-selected epoch {sel}: val overall {v['overall']:.2f} / open {v['open']:.2f} / closed {v['closed']:.2f} (test not evaluated)"
    fig.suptitle(f'{title}  |  {head}', fontsize=11)
    # x 축 눈금: epoch 가 15 이상이면 5 단위, 그보다 짧으면 1 단위
    last = max([max(val)] + ([max(test)] if test else []) + ([max(train_loss)] if train_loss else []))
    step = 5 if last >= 15 else 1
    for a in ax.ravel():
        a.xaxis.set_major_locator(MultipleLocator(step)); a.xaxis.set_major_formatter(FormatStrFormatter('%d'))
        a.grid(axis='x', alpha=0.2)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, sel


In [ ]:
# C-14. 학습 곡선 (선택 = val). test 는 val 로 고른 epoch(best)와 마지막 epoch(last)만 평가했으므로 두 점만 표시한다.
import sys
sys.path.insert(0, '/content')
import importlib, curve_plot; importlib.reload(curve_plot)      # 저장 셀로 갱신한 최신 코드를 쓰도록 다시 불러온다
from curve_plot import plot_curves
_recs = [json.loads(l) for l in open(f"{SEL['dir']}/val_metrics.jsonl")]
_ep = lambda r: int(round(r['epoch']))
val  = {_ep(r): {'open': r['open_recall'], 'closed': r['closed'], 'overall': r['overall']} for r in _recs}
loss = {_ep(r): r['train_loss'] for r in _recs if r.get('train_loss') is not None}
sel = int(round(SEL['epoch']))
test = {}
if res.get('best'): test[sel] = {k: res['best'][k] for k in ('open', 'closed', 'overall')}
if res.get('last'): test[max(val)] = {k: res['last'][k] for k in ('open', 'closed', 'overall')}
fig, _ = plot_curves(f"LLaVA-Med v1.0 + QLoRA (SLAKE, lr {SEL['lr']:g}, LoRA+mm_projector)", train_loss=loss, val=val, test=test,
                     paper=paper_slake, out_png=f'{QROOT}/curves.png', selected=sel)
print('저장:', f'{QROOT}/curves.png')


In [ ]:
# C-15. closed(yes/no) 혼동행렬 + precision / recall / F1 — val 로 고른 체크포인트(best)의 test 예측 (ROC 는 확률 점수가 없어 그리지 않는다)
import importlib, curve_plot; importlib.reload(curve_plot)
from curve_plot import plot_closed_f1
_refs = {int(k): v for k, v in json.load(open(f'{EVAL_DIR}/slake_ref.json')).items()}
_preds = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/slake_qlora_best_pred.jsonl')}
assert set(_preds) == set(_refs), '예측과 참조의 id 가 다릅니다 (추론이 끝났는지 확인)'
def _cls(text):
    tk = normalize(text); y, n = 'yes' in tk, 'no' in tk
    return 'yes' if y and not n else 'no' if n and not y else 'other'
_ids = [i for i, r in _refs.items() if r['answer_type'] == 'CLOSED' and r['answer'].strip().lower() in ('yes', 'no')]
_n_closed = sum(r['answer_type'] == 'CLOSED' for r in _refs.values())
fig, m = plot_closed_f1(f"LLaVA-Med + QLoRA SLAKE closed yes/no (n={len(_ids)} of {_n_closed} closed)",
                        [_refs[i]['answer'].strip().lower() for i in _ids], [_cls(_preds[i]) for i in _ids], out_png=f'{QROOT}/closed_f1.png')
print(f"accuracy {m['accuracy']:.2f} | macro F1 {m['macro']['f1']:.2f} | yes F1 {m['yes']['f1']:.2f} (n={m['yes']['support']}) | no F1 {m['no']['f1']:.2f} (n={m['no']['support']})")
print(f'※ closed 문항 중 정답이 yes/no 가 아닌 {_n_closed - len(_ids)}개는 F1 에서 제외했다 (원 채점에서는 이런 문항이 항상 0점)')


## C. 종합 — Open / Closed / ALL (원본 채점 vs S1+S2)

In [6]:
# 종합. Open / Closed / ALL — 원본 채점 vs S1+S2 + 논문 대비 격차 (VQA-RAD 와 SLAKE 를 한 번에)
# 필요: A 의 추론이 끝났을 것(vqarad_pred_greedy.jsonl) / B 의 C-10 이 끝났을 것(slake_qlora_best_pred.jsonl). 없는 쪽은 건너뛴다.
# 원본 = Phi 방식 (closed: 정답 단어 포함, open: 단어 recall). S1+S2 = open 만 보조 규칙 적용 (규칙이 Phi 의 VQA-RAD test 오답에서 만들어져 규칙 완화 — 항상 원본 행과 같이 보고).
import os, sys, json
if '/content/Expert-CFG' not in sys.path: sys.path.insert(0, '/content/Expert-CFG')
import vqa_scoring as vs
PAPER = {'VQA-RAD': {'open': 61.52, 'closed': 84.19, 'overall': 75.19},
         'SLAKE':   {'open': 83.08, 'closed': 85.34, 'overall': 83.97}}
JOBS = [('VQA-RAD', f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl', '공식 delta · greedy · 접미어'),
        ('SLAKE',   f'{EVAL_DIR}/slake_ref.json',  f'{DRIVE_ROOT}/slake_qlora_best_pred.jsonl', 'QLoRA 근사 · val 선택')]
RECS = {}
print(f"{'데이터셋':9s}{'채점':>7s}{'open':>8s}{'closed':>8s}{'ALL':>8s}{'논문 ALL':>10s}{'격차':>8s}   조건")
for name, ref, pred, cond in JOBS:
    if not (os.path.exists(ref) and os.path.exists(pred)):
        print(f'{name:9s} 건너뜀 — ref 있음: {os.path.exists(ref)} / 예측 있음: {os.path.exists(pred)}'); continue
    RECS[name] = vs.load_llava(ref, pred)
    for lab, sc in (('원본', vs.score_phi), ('S1+S2', vs.score_s1s2)):
        m = vs.summarize(RECS[name], sc)
        print(f"{name:9s}{lab:>7s}{m['open']:8.2f}{m['closed']:8.2f}{m['overall']:8.2f}{PAPER[name]['overall']:10.2f}{m['overall'] - PAPER[name]['overall']:+8.2f}   {cond}")
print('\n※ 판정(2%p 이내)은 원본 행 기준으로 먼저 보고, S1+S2 는 별도 행으로 표기한다.')
for name, rec in RECS.items():
    vs.print_ci(rec, PAPER[name], f'LLaVA-Med {name}', vs.score_phi)


데이터셋          채점    open  closed     ALL    논문 ALL      격차   조건
VQA-RAD       원본   66.08   72.06   69.69     75.19   -5.50   공식 delta · greedy · 접미어
VQA-RAD    S1+S2   66.91   72.06   70.02     75.19   -5.17   공식 delta · greedy · 접미어
SLAKE         원본   81.77   75.72   79.40     83.97   -4.57   QLoRA 근사 · val 선택
SLAKE      S1+S2   81.77   75.72   79.40     83.97   -4.57   QLoRA 근사 · val 선택

※ 판정(2%p 이내)은 원본 행 기준으로 먼저 보고, S1+S2 는 별도 행으로 표기한다.
LLaVA-Med VQA-RAD  (n=451, 문항 복원추출 10000회, 95% 구간)
               점수      하한      상한      논문  논문이 구간 안?
open        66.08   59.57   72.49   61.52  예
closed      72.06   66.54   77.21   84.19  아니오
overall     69.69   65.43   73.79   75.19  아니오
LLaVA-Med SLAKE  (n=1061, 문항 복원추출 10000회, 95% 구간)
               점수      하한      상한      논문  논문이 구간 안?
open        81.77   78.97   84.52   83.08  예
closed      75.72   71.63   79.81   85.34  아니오
overall     79.40   77.07   81.71   83.97  아니오


In [17]:
# 채점용 ref 만 다시 만들기 (GPU · 이미지 · 추론 불필요) — 런타임이 초기화돼 /content/llava_eval_data 가 사라졌을 때
# 원래 평가 데이터 셀과 같은 필터·같은 순서로 만들어야 예측 파일의 question_id(0부터 순번)와 맞는다.
import os, json, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/expert_cfg_llavamed'
EVAL_DIR = '/content/llava_eval_data'
os.makedirs(EVAL_DIR, exist_ok=True)

def write_ref(name, rows):
    refs = {i: {'answer': str(r['answer']), 'answer_type': r['answer_type']} for i, r in enumerate(rows)}
    json.dump(refs, open(f'{EVAL_DIR}/{name}_ref.json', 'w'))
    n_open = sum(v['answer_type'] == 'OPEN' for v in refs.values())
    print(f'{name}_ref.json: {len(rows)}문항 (OPEN {n_open} / CLOSED {len(rows) - n_open})')
    return refs

# VQA-RAD: 원본 공개 JSON 의 test 문항(phrase_type 이 test_freeform / test_para), 파일 순서 그대로
_raw = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
if not os.path.exists(_raw):
    _raw = f'{EVAL_DIR}/VQA_RAD_Dataset_Public.json'
    subprocess.run(['wget', '-q', '-O', _raw, 'https://osf.io/download/9dpze/'], timeout=120, check=True)
all_data = json.load(open(_raw))
for it in all_data:
    it['answer_type'] = it.get('answer_type', 'CLOSED' if str(it.get('answer', '')).strip().lower() in ('yes', 'no') else 'OPEN').upper()
vq = [r for r in all_data if str(r.get('phrase_type', '')).lower() in ('test_freeform', 'test_para')]
assert len(vq) == 451, len(vq)
ref_v = write_ref('vqarad', vq)

# SLAKE: HF BoKelvin/SLAKE test.json 의 영어 문항, 파일 순서 그대로
from huggingface_hub import hf_hub_download
sl = [r for r in json.load(open(hf_hub_download('BoKelvin/SLAKE', 'test.json', repo_type='dataset'))) if r['q_lang'] == 'en']
for r in sl: r['answer_type'] = r['answer_type'].upper()
assert len(sl) == 1061, len(sl)
ref_s = write_ref('slake', sl)

# 예측 파일과 대응 확인: 줄 수 · question_id 집합
for name, ref, pf in (('VQA-RAD', ref_v, 'vqarad_pred_greedy.jsonl'), ('SLAKE', ref_s, 'slake_qlora_best_pred.jsonl')):
    pr = [json.loads(l) for l in open(f'{DRIVE_ROOT}/{pf}')]
    ids = {int(x['question_id']) for x in pr}
    print(f'{name}: 예측 {len(pr)}줄, question_id 가 ref 와 같은 집합: {ids == set(ref)}')


vqarad_ref.json: 451문항 (OPEN 179 / CLOSED 272)
slake_ref.json: 1061문항 (OPEN 645 / CLOSED 416)
VQA-RAD: 예측 451줄, question_id 가 ref 와 같은 집합: True
SLAKE: 예측 1061줄, question_id 가 ref 와 같은 집합: True


In [18]:
# A 채점 — LLaVA-Med 자체 평가 코드(microsoft/LLaVA-Med v1.0.0 run_eval.py) 이식본으로 재채점 (GPU 불필요)
# 논문 재현 판정(A)에는 이 모델의 원래 평가 방식을 쓴다: closed = 예측에 yes/no 가 있고 정답이 정규화된 예측의 부분문자열이면 정답,
# open = 정규화 후 토큰 recall. 위 종합 셀의 phi 채점과 나란히 둔다. 이식하지 않은 것: contractions 사전(영향 미확인).
# 주의: SLAKE 행은 공식 delta 가 없어 QLoRA 근사 학습이라 A(원래 프로토콜)로 부를 수 없다 — 근사 행으로만 표기한다.
import os, sys, importlib
if '/content/Expert-CFG' not in sys.path: sys.path.insert(0, '/content/Expert-CFG')
import vqa_scoring as vs; importlib.reload(vs)
assert hasattr(vs, 'score_llava'), 'vqa_scoring.py 가 옛 버전입니다 — 새 파일을 /content/Expert-CFG 에 올리세요'
import pandas as pd
from IPython.display import display
PAPER_L = {'VQA-RAD': (61.52, 84.19, 75.19), 'SLAKE': (83.08, 85.34, 83.97)}      # open, closed, overall
rows_ = []
for name, ref, pred, cond in (('VQA-RAD', f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl', '공식 delta · greedy · 접미어 (A)'),
                              ('SLAKE', f'{EVAL_DIR}/slake_ref.json', f'{DRIVE_ROOT}/slake_qlora_best_pred.jsonl', 'QLoRA 근사 (A 아님)')):
    if not (os.path.exists(ref) and os.path.exists(pred)):
        print('건너뜀:', name); continue
    rec = vs.load_llava(ref, pred)
    p_ = PAPER_L[name]
    for lab, sc in (('llava 자체 평가 (A)', vs.score_llava), ('phi (참고)', vs.score_phi)):
        m = vs.summarize(rec, sc)
        rows_.append({'데이터셋': name, '조건': cond, '채점': lab, 'open': m['open'], 'closed': m['closed'], 'overall': m['overall'],
                      'Δopen': m['open'] - p_[0], 'Δclosed': m['closed'] - p_[1], 'Δoverall': m['overall'] - p_[2]})
    cl = [r for r in rec if r['answer_type'] == 'CLOSED']
    nonyn = [r for r in cl if r['answer'].strip().lower() not in ('yes', 'no')]
    print(f"{name}: closed {len(cl)}문항 중 정답이 yes/no 가 아닌 것 {len(nonyn)}개 — llava 규칙에서 정답으로 친 것 {int(sum(vs.score_llava(r) for r in nonyn))}개 / phi 규칙 {int(sum(vs.score_phi(r) for r in nonyn))}개")
if rows_:
    print('\n■ LLaVA-Med 논문 대비 (논문 VQA-RAD 61.52 / 84.19 / 75.19, SLAKE 83.08 / 85.34 / 83.97)')
    display(pd.DataFrame(rows_).style.format({c_: '{:.2f}' for c_ in ('open', 'closed', 'overall')} | {c_: '{:+.2f}' for c_ in ('Δopen', 'Δclosed', 'Δoverall')}).hide(axis='index'))


VQA-RAD: closed 272문항 중 정답이 yes/no 가 아닌 것 21개 — llava 규칙에서 정답으로 친 것 0개 / phi 규칙 4개
SLAKE: closed 416문항 중 정답이 yes/no 가 아닌 것 61개 — llava 규칙에서 정답으로 친 것 0개 / phi 규칙 36개

■ LLaVA-Med 논문 대비 (논문 VQA-RAD 61.52 / 84.19 / 75.19, SLAKE 83.08 / 85.34 / 83.97)


데이터셋,조건,채점,open,closed,overall,Δopen,Δclosed,Δoverall
VQA-RAD,공식 delta · greedy · 접미어 (A),llava 자체 평가 (A),65.70,70.59,68.65,+4.18,-13.60,-6.54
VQA-RAD,공식 delta · greedy · 접미어 (A),phi (참고),66.08,72.06,69.69,+4.56,-12.13,-5.50
SLAKE,QLoRA 근사 (A 아님),llava 자체 평가 (A),81.89,67.07,76.08,-1.19,-18.27,-7.89
SLAKE,QLoRA 근사 (A 아님),phi (참고),81.77,75.72,79.40,-1.31,-9.62,-4.57


In [5]:
import json
from huggingface_hub import hf_hub_download
slake_test = [r for r in json.load(open(hf_hub_download('BoKelvin/SLAKE', 'test.json', repo_type='dataset'))) if r['q_lang'] == 'en']
assert len(slake_test) == 1061, len(slake_test)
for r in slake_test: r['answer_type'] = r['answer_type'].upper()
write_eval_files('slake', slake_test, lambda r: r['img_name'])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


test.json:   0%|          | 0.00/636k [00:00<?, ?B/s]

slake: 1061문항 (OPEN 645 / CLOSED 416)


In [19]:
# SLAKE 학습 메타 확인 (GPU 불필요) — 보고한 79.40 행이 몇 epoch 학습에서, 어느 체크포인트로 나왔는지 Drive 기록으로 확인
# 노트북 설정 셀의 FINAL_EPOCHS(=9)와 실제 실행(6 epoch 로 보고됨)이 달라서 둔다. 이 셀은 읽기만 한다.
# 참고: SLAKE 영어 train 4,919문항, 유효 배치 16(4 x 누적 4) → epoch 당 약 307 optimizer step, 6 epoch = 1842, 9 epoch = 2763
import os, re, json, glob
QROOT = f'{DRIVE_ROOT}/qlora_slake'
print('QROOT:', QROOT, '| 존재:', os.path.isdir(QROOT))
for stage_dir in sorted(glob.glob(f'{QROOT}/*/lr*')):
    name = os.path.relpath(stage_dir, QROOT)
    ck = sorted(glob.glob(f'{stage_dir}/checkpoint-*'), key=lambda d: int(d.rsplit('-', 1)[1]))
    print(f'\n[{name}] 체크포인트: {[os.path.basename(c) for c in ck]}')
    for c in ck:
        ts = f'{c}/trainer_state.json'
        if os.path.exists(ts):
            t = json.load(open(ts))
            print(f"   {os.path.basename(c)}: global_step={t.get('global_step')} epoch={t.get('epoch')} max_steps={t.get('max_steps')} num_train_epochs={t.get('num_train_epochs')}")
    ri = f'{stage_dir}/run_info.json'
    if os.path.exists(ri):
        print('   run_info:', json.load(open(ri)))
    vm = f'{stage_dir}/val_metrics.jsonl'
    if os.path.exists(vm):
        rows = [json.loads(l) for l in open(vm)]
        print(f'   val_metrics: {len(rows)}행, epoch {[round(r["epoch"], 2) for r in rows]}')
        b = max(rows, key=lambda r: (r['overall'], -r['eval_loss']))
        print(f"   val overall 최고: epoch {b['epoch']:.0f}  overall {b['overall']:.2f}  closed {b['closed']:.2f}  open recall {b['open_recall']:.2f}")
    lg = f'{stage_dir}/stdout.log'
    if os.path.exists(lg):
        txt = open(lg, errors='ignore').read()
        m = re.findall(r'[^\n]*(?:epochs|num_train_epochs)[^\n]*', txt)[:3]
        print('   로그의 epochs 관련 줄:', [x.strip()[:140] for x in m])
for f in glob.glob(f'{DRIVE_ROOT}/slake_qlora_*'):
    print('\n예측 파일:', os.path.basename(f), os.path.getsize(f), 'bytes')


QROOT: /content/drive/MyDrive/expert_cfg_llavamed/qlora_slake | 존재: True

[final/lr0.0002] 체크포인트: []
   로그의 epochs 관련 줄: []

[final_proj/lr0.0002] 체크포인트: ['checkpoint-1537', 'checkpoint-1842']
   checkpoint-1537: global_step=1537 epoch=4.998373983739837 max_steps=1842 num_train_epochs=6
   checkpoint-1842: global_step=1842 epoch=5.990243902439024 max_steps=1842 num_train_epochs=6
   run_info: {'lr': 0.0002, 'epochs': 6.0, 'max_steps': -1, 'best_checkpoint': '/content/drive/MyDrive/expert_cfg_llavamed/qlora_slake/final_proj/lr0.0002/checkpoint-1842', 'last_checkpoint': '/content/drive/MyDrive/expert_cfg_llavamed/qlora_slake/final_proj/lr0.0002/checkpoint-1842', 'best_val_overall': 80.29813807591587, 'n_train': 4919, 'n_val': 1053, 'train_projector': True, 'proj_lr': 2e-05, 'grad_ckpt': False, 'peak_gpu_mem_gb': 24.3}
   val_metrics: 6행, epoch [1.0, 2.0, 3.0, 4.0, 5.0, 5.99]
   val overall 최고: epoch 6  overall 80.30  closed 74.64  open recall 84.08
   로그의 epochs 관련 줄: ['DONE {"lr": 0.000

In [20]:
import os, re, json
import pandas as pd
from IPython.display import display

DRIVE_ROOT = '/content/drive/MyDrive/expert_cfg_llavamed'

def cls(text):
    t = re.sub(r'[^a-z\s]', ' ', str(text).lower()).split()
    y, n = 'yes' in t, 'no' in t
    return 'yes' if (y and not n) else 'no' if (n and not y) else '둘다' if (y and n) else '없음'

def analyze(name, ref_file, pred_file):
    refs = {int(k): v for k, v in json.load(open(ref_file)).items()}
    preds = {}
    for line in open(pred_file):
        d = json.loads(line); preds[int(d['question_id'])] = d
    rows = []
    for i, r in refs.items():
        if r['answer_type'] != 'CLOSED': continue
        gt = r['answer'].strip().lower(); txt = preds[i]['text']
        rows.append({'i': i, '정답': gt, '정답형': 'yes/no' if gt in ('yes', 'no') else '그 외', '예측분류': cls(txt),
                     '질문': re.sub(r'\s*Please choose.*$', '', preds[i].get('prompt', ''))[:70], '예측': txt.strip()[:90]})
    df = pd.DataFrame(rows)
    print(f'\n===== {name}: closed {len(df)}문항 =====')
    yn = df[df['정답형'] == 'yes/no']
    print(f'정답이 yes/no 인 {len(yn)}문항: yes {int((yn.정답 == "yes").sum())} / no {int((yn.정답 == "no").sum())}')
    print('\n■ 혼동표 (행=정답, 열=예측에서 읽은 yes/no)'); display(pd.crosstab(yn['정답'], yn['예측분류']))
    ans = yn[yn.예측분류.isin(['yes', 'no'])]
    print(f'yes/no 로 답한 {len(ans)}문항 중 맞음 {int((ans.정답 == ans.예측분류).sum())} ({100 * (ans.정답 == ans.예측분류).mean():.1f}%) | yes/no 를 못 읽은 {int((~yn.예측분류.isin(["yes", "no"])).sum())}문항')
    print(f'정답 yes/no 문항 기준 정확도 {100 * (yn.정답 == yn.예측분류).mean():.1f}% | 예측 yes 비율 {100 * (yn.예측분류 == "yes").mean():.1f}% (정답 yes 비율 {100 * (yn.정답 == "yes").mean():.1f}%)')
    for _, r in yn[~yn.예측분류.isin(['yes', 'no'])].head(6).iterrows(): print(f"  [형식이탈] Q: {r['질문']} | 정답: {r['정답']} | 예측: {r['예측']}")
    oth = df[df['정답형'] == '그 외']
    print(f'\n■ 정답이 yes/no 가 아닌 closed {len(oth)}문항')
    for _, r in oth.head(8).iterrows(): print(f"  Q: {r['질문']} | 정답: {r['정답']} | 예측: {r['예측']}")

EV = '/content/llava_eval_data'
for name, ref, pf in (('VQA-RAD', f'{EV}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'),
                      ('SLAKE', f'{EV}/slake_ref.json', f'{DRIVE_ROOT}/slake_qlora_best_pred.jsonl')):
    analyze(name, ref, pf)



===== VQA-RAD: closed 272문항 =====
정답이 yes/no 인 251문항: yes 118 / no 133

■ 혼동표 (행=정답, 열=예측에서 읽은 yes/no)


예측분류,no,yes
정답,,
no,83,50
yes,9,109


yes/no 로 답한 251문항 중 맞음 192 (76.5%) | yes/no 를 못 읽은 0문항
정답 yes/no 문항 기준 정확도 76.5% | 예측 yes 비율 63.3% (정답 yes 비율 47.0%)

■ 정답이 yes/no 가 아닌 closed 21문항
  Q: Are there multiple or just 1 metastatic focus? Please choose from the  | 정답: one | 예측: Assistant: no
  Q: Is the colon more prominent on the patient's right or left side? Pleas | 정답: left | 예측: Assistant: Left side
  Q: Is the heart size in this image smaller or larger than if the image wa | 정답: smaller | 예측: Assistant: yes
  Q: Is this supratentorial or infratentorial? Please choose from the follo | 정답: supratentorial | 예측: Assistant: supratentorial
  Q: Is this an MRI or a CT scan? Please choose from the following two opti | 정답: mri | 예측: Assistant: no
  Q: Is the lesion on the left or right side of the brain? Please choose fr | 정답: left | 예측: Assistant: Right
  Q: Is the patient a female or male? Please choose from the following two  | 정답: female | 예측: Assistant: no
  Q: Is this patient male or female? Please choose from the followi

예측분류,no,yes
정답,,
no,143,37
yes,39,136


yes/no 로 답한 355문항 중 맞음 279 (78.6%) | yes/no 를 못 읽은 0문항
정답 yes/no 문항 기준 정확도 78.6% | 예측 yes 비율 48.7% (정답 yes 비율 49.3%)

■ 정답이 yes/no 가 아닌 closed 61문항
  Q: Which is the biggest in this image,lung,liver or heart? Please choose  | 정답: lung | 예측: Assistant: Lung
  Q: Which is the biggest in this image,lung,liver or heart? Please choose  | 정답: lung | 예측: Assistant: Lung
  Q: Which organ is abnormal, heart or lung? Please choose from the followi | 정답: heart | 예측: Assistant: Lung
  Q: Which organ is abnormal, heart or lung? Please choose from the followi | 정답: heart | 예측: Assistant: Lung
  Q: Which organ is abnormal, heart or lung? Please choose from the followi | 정답: heart | 예측: Assistant: Lung
  Q: Which organ is abnormal, heart or lung? Please choose from the followi | 정답: lung | 예측: Assistant: Lung
  Q: Which organ is abnormal, heart or lung? Please choose from the followi | 정답: heart | 예측: Assistant: Lung
  Q: Which organ is abnormal, heart or lung? Please choose from the followi | 정답: hea

## 부록 A — closed 개선 탐색 (test 기반, **규칙 완화 행**)

기준 행 추론(`vqarad_pred_greedy.jsonl`)이 있어야 한다. 아래 두 셀은 test 로 프롬프트·규칙을 골랐으므로 **규칙 준수 행과 별도 행으로 표기**한다.
- **C-17**: 접미어를 정답 유형이 아니라 질문 문장으로 판단해 붙이는 프롬프트 변형.
- **C-19**: yes/no 로그확률의 임계값 보정. `oracle` 열은 상한 참고용이며 보고하지 않는다.

In [ ]:
# C-17. VQA-RAD closed 프롬프트 변형 (test 기반 탐색 — 규칙 완화, 별도 행으로 표기할 것).
# 접미어 "[yes, no]" 를 정답 유형이 아니라 '질문 문장'으로 판단해 붙인다: yes/no 형태 질문에만 붙이고, 선택지가 있는 질문("left or right?" 등)은 접미어 없이 원문 그대로 묻는다.
# 이미 있는 greedy 예측(vqarad_pred_greedy.jsonl)을 재사용하고, 접미어를 빼는 문항만 다시 추론하므로 빠르다. 모델 로딩 때문에 1~2분 정도 걸린다(추정).
import json, re, os
from collections import Counter
import numpy as np
if 'score_jsonl' not in globals():      # 채점 함수 셀을 실행하지 않았어도 이 셀만으로 채점되도록 같은 정의를 둔다 (본문 채점 셀과 동일한 방식)
    def normalize(text):
        text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
        return re.sub(r'[^a-z0-9\s]', ' ', text).split()
    def closed_accuracy(pred, gt):
        return float(gt.strip().lower() in normalize(pred))
    def open_recall(pred, gt):
        g, p = Counter(normalize(gt)), Counter(normalize(pred))
        return sum((g & p).values()) / sum(g.values()) if g else 0.0
    def score_jsonl(ref_file, pred_file):
        refs = {int(k): v for k, v in json.load(open(ref_file)).items()}
        preds = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(pred_file)}
        assert set(preds) == set(refs), f'예측 {len(preds)}개 / 참조 {len(refs)}개'
        cl, op = [], []
        for i, r in refs.items():
            (cl if r['answer_type'] == 'CLOSED' else op).append(closed_accuracy(preds[i], r['answer']) if r['answer_type'] == 'CLOSED' else open_recall(preds[i], r['answer']))
        return {'open': 100 * np.mean(op), 'closed': 100 * np.mean(cl), 'overall': 100 * np.mean(cl + op), 'n_open': len(op), 'n_closed': len(cl)}
SUF = ' Please choose from the following two options: [yes, no]'
AUX = ('is', 'are', 'was', 'were', 'does', 'do', 'did', 'can', 'could', 'has', 'have', 'had', 'will', 'would', 'should')
def is_yesno_form(q):
    q = q.strip().lower()
    return q.split()[0].strip(',') in AUX and not re.search(r'\bor\b', q)

qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))
ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
redo = []
for q in qs:
    if ref[str(q['id'])]['answer_type'] != 'CLOSED': continue
    v = q['conversations'][0]['value']
    if v.endswith(SUF): v = v[:-len(SUF)]
    if not is_yesno_form(v):
        q = json.loads(json.dumps(q)); q['conversations'][0]['value'] = v; redo.append(q)
json.dump(redo, open(f'{EVAL_DIR}/vqarad_closed_qform.json', 'w'))
# 분류가 맞았는지 확인용 (정답은 채점·점검에만 사용, 프롬프트 결정에는 쓰지 않음)
_gt = Counter('yes/no' if ref[str(q['id'])]['answer'].strip().lower() in ('yes', 'no') else 'other' for q in redo)
print(f'접미어 없이 다시 묻는 closed 문항: {len(redo)}개 (정답이 yes/no 인 것 {_gt["yes/no"]}, 아닌 것 {_gt["other"]})')
for q in redo[:8]: print('  ', q['conversations'][0]['value'][:90])

# 추론 스크립트가 greedy 가 아니면(do_sample=True, temperature 0.7) 기준 예측(greedy)과 비교가 어긋나므로 먼저 greedy 로 패치한다
_sp = '/content/LLaVA-Med/llava/eval/model_vqa_med.py'
_s = open(_sp).read()
_pat = r"do_sample=True,\s*temperature=0\.7,\s*max_new_tokens=1024,"
if re.search(_pat, _s):
    open(_sp, 'w').write(re.sub(_pat, "do_sample=False,\n                max_new_tokens=1024,", _s, count=1))
    print('추론 스크립트를 greedy 로 패치했습니다')
else:
    print('추론 스크립트: 이미 greedy')
%cd /content/LLaVA-Med
_out = f'{DRIVE_ROOT}/vqarad_pred_closed_qform.jsonl'
if os.path.exists(_out): os.remove(_out)
!PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_closed_qform.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {_out} > /content/vqarad_qform.log 2>&1

new = [json.loads(l) for l in open(_out)]
assert len(new) == len(redo), f'예측 {len(new)}/{len(redo)} — /content/vqarad_qform.log 를 확인하세요'
old = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl')]
ids = {p['question_id'] for p in new}
with open(f'{DRIVE_ROOT}/vqarad_pred_qform.jsonl', 'w') as f:
    for p in [p for p in old if p['question_id'] not in ids] + new: f.write(json.dumps(p) + '\n')
print('\n기준(전 문항 접미어, greedy):', score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'))
print('변형(질문 형태 기준 접미어)  :', score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_qform.jsonl'))
print('※ test 기반 탐색 결과입니다. 규칙 준수 행과 별도 행으로 표기하세요.')


In [ ]:
# C-19. VQA-RAD yes/no 문항의 편향 보정 (test 기반 탐색 — 규칙 완화, 별도 행). 추론 + 분석을 한 셀에서 한다. C-17 을 먼저 (greedy 로) 실행해 둘 것.
# 모델이 "Assistant:" 다음에 내는 첫 답 토큰의 yes/no 로그확률 차이 d = log P(yes) - log P(no) 를 구한다.
# 선택지 순서를 [yes, no] / [no, yes] 로 바꾼 두 번을 물어 평균한다 (순서 편향 완화). 임계값 규칙 3가지를 비교한다:
#   0      : d > 0 이면 yes (보정 없음, 모델 자체 선호)
#   prior  : 예측 yes 비율이 train 의 yes 비율과 같아지도록 임계값을 정한다 (test 정답을 쓰지 않음)
#   oracle : test 정답으로 고른 최적 임계값 (상한 참고용 — 보고하지 말 것)
import json, re, os, subprocess
import numpy as np
from collections import Counter
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/expert_cfg_llavamed')
EVAL_DIR   = globals().get('EVAL_DIR', '/content/llava_eval_data')
PY         = globals().get('PY', '/content/venv_llava/bin/python')
IMG_DIR    = globals().get('VQARAD_IMG_DIR', '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder')
PUBLIC     = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
SUF_A = ' Please choose from the following two options: [yes, no]'
SUF_B = ' Please choose from the following two options: [no, yes]'
AUX = ('is', 'are', 'was', 'were', 'does', 'do', 'did', 'can', 'could', 'has', 'have', 'had', 'will', 'would', 'should')
def is_yesno_form(q):
    q = q.strip().lower()
    return q.split()[0].strip(',') in AUX and not re.search(r'\bor\b', q)
norm = lambda s: re.sub(r'[^a-z0-9\s]', ' ', str(s).lower().replace('<|end|>', ' ').replace('###', ' ')).split()

# 1) 질문 파일: closed 중 yes/no 형태 문항, 두 가지 선택지 순서 (순서 B 는 id + 100000)
qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json')); ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
ynq, base_ids = [], []
for q in qs:
    if ref[str(q['id'])]['answer_type'] != 'CLOSED': continue
    v = q['conversations'][0]['value']
    if v.endswith(SUF_A): v = v[:-len(SUF_A)]
    if not is_yesno_form(v): continue
    base_ids.append(q['id'])
    for k, suf in enumerate((SUF_A, SUF_B)):
        qq = json.loads(json.dumps(q)); qq['id'] = q['id'] + 100000 * k; qq['conversations'][0]['value'] = v + suf; ynq.append(qq)
json.dump(ynq, open(f'{EVAL_DIR}/vqarad_yn2.json', 'w'))
print(f'yes/no 형태 closed 문항 {len(base_ids)}개 x 2가지 순서 = {len(ynq)}회 추론')

# 2) 추론 스크립트 복사본: greedy + 첫 답 토큰의 yes/no 로그확률을 .yn 파일에 기록
_src = open('/content/LLaVA-Med/llava/eval/model_vqa_med.py').read()
_pat = re.compile(r"with torch\.inference_mode\(\):\s*output_ids = model\.generate\(.*?stopping_criteria=\[stopping_criteria\]\)\n", re.S)
_n = len(_pat.findall(_src))
if _n == 0:      # 형태가 다르면 generate 주변을 그대로 보여 준다
    _k = _src.find('model.generate')
    print(_src[max(0, _k - 200):_k + 500] if _k >= 0 else 'model.generate 가 없습니다')
    print('CRLF 줄바꿈:', chr(13) in _src)
assert _n >= 1, 'model_vqa_med.py 의 generate 블록을 찾지 못했습니다 (위 출력을 알려 주세요)'
print(f'generate 블록 {_n}개 발견 -> 첫 번째만 교체')
_new = '''with torch.inference_mode():
            _go = model.generate(input_ids, images=images, do_sample=False, max_new_tokens=24,
                                 stopping_criteria=[stopping_criteria], output_scores=True, return_dict_in_generate=True)
        output_ids = _go.sequences
        _gen = output_ids[0, input_ids.shape[1]:]
        _t = 0
        for _k in range(len(_gen)):      # 'Assistant:' 이후의 첫 토큰 위치를 찾는다
            if ':' in tokenizer.decode(_gen[:_k + 1], skip_special_tokens=True):
                _t = _k + 1
                break
        _vy = [tokenizer.convert_tokens_to_ids(w) for w in ('yes', 'Yes', chr(9601) + 'yes', chr(9601) + 'Yes')]
        _vn = [tokenizer.convert_tokens_to_ids(w) for w in ('no', 'No', chr(9601) + 'no', chr(9601) + 'No')]
        _vy = [x for x in _vy if x is not None and x != tokenizer.unk_token_id]
        _vn = [x for x in _vn if x is not None and x != tokenizer.unk_token_id]
        _ly = _ln = None
        if _t < len(_go.scores):
            _lp = torch.log_softmax(_go.scores[_t][0].float(), -1)
            _ly = torch.logsumexp(_lp[_vy], 0).item()
            _ln = torch.logsumexp(_lp[_vn], 0).item()
        open(args.answers_file + '.yn', 'a').write(json.dumps({'question_id': idx, 'lp_yes': _ly, 'lp_no': _ln, 't': _t,
                                                              'head': tokenizer.decode(_gen[:_t + 3], skip_special_tokens=True)}) + chr(10))
'''
_YN = '/content/LLaVA-Med/llava/eval/model_vqa_yn.py'
open(_YN, 'w').write(_pat.sub(lambda m: _new, _src, count=1))

_out = f'{DRIVE_ROOT}/vqarad_pred_yn2.jsonl'
for f in (_out, _out + '.yn'):
    if os.path.exists(f): os.remove(f)
%cd /content/LLaVA-Med
import time
_cmd = (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_yn.py --model-name /content/llava-med-7b-vqarad '
        f'--question-file {EVAL_DIR}/vqarad_yn2.json --image-folder "{IMG_DIR}" --answers-file {_out} > /content/vqarad_yn2.log 2>&1')
_proc = subprocess.Popen(['bash', '-c', _cmd])
_t0 = time.time()
while _proc.poll() is None:      # 진행 상황: 로그확률이 기록된 문항 수 (1분마다 출력)
    _n = sum(1 for _ in open(_out + '.yn')) if os.path.exists(_out + '.yn') else 0
    print(f'[{(time.time() - _t0) / 60:5.1f}분] 진행 {_n}/{len(ynq)}', flush=True)
    time.sleep(60)
assert _proc.returncode == 0, f'추론 실패 (종료 코드 {_proc.returncode}) — /content/vqarad_yn2.log 의 끝부분을 확인하세요'
print(f'추론 완료 ({(time.time() - _t0) / 60:.1f}분)')

# 3) 분석
yn = {}
for l in open(_out + '.yn'):
    r = json.loads(l); yn[r['question_id']] = r
bad = [q['id'] for q in ynq if q['id'] not in yn or yn[q['id']]['lp_yes'] is None]
assert not bad, f'로그확률이 없는 문항 {len(bad)}개 — /content/vqarad_yn2.log 확인. 예: {bad[:5]}'
print('첫 답 위치 확인 (t: 답 토큰 위치, head: 모델 출력 앞부분):')
for q in ynq[:4]: print('  ', yn[q['id']]['t'], repr(yn[q['id']]['head']))
dA = np.array([yn[i]['lp_yes'] - yn[i]['lp_no'] for i in base_ids])
dB = np.array([yn[i + 100000]['lp_yes'] - yn[i + 100000]['lp_no'] for i in base_ids])
gt = np.array([ref[str(i)]['answer'].strip().lower() for i in base_ids])
isyn = np.isin(gt, ['yes', 'no']); gy = (gt == 'yes')

pub = json.load(open(PUBLIC))
_tr = [r for r in pub if str(r.get('phrase_type', '')).lower() not in ('test_freeform', 'test_para') and str(r.get('answer', '')).strip().lower() in ('yes', 'no')]
p_yes = np.mean([str(r['answer']).strip().lower() == 'yes' for r in _tr])
print(f'train 의 yes 비율 {p_yes:.3f} (yes/no 정답 {len(_tr)}개) | 테스트 yes/no 문항 중 정답 yes 비율 {gy[isyn].mean():.3f}')

def acc(pred_yes): return int(((pred_yes == gy) & isyn).sum())          # 정답이 yes/no 가 아닌 문항은 항상 오답
srcs = {'순서A': dA, '순서B': dB, '평균': (dA + dB) / 2}
rules = {}
for name, d in srcs.items():
    cand = np.sort(d)
    best = max(acc(d > (a + b) / 2) for a, b in zip(cand[:-1], cand[1:]))
    rules[name] = {'0': acc(d > 0), 'prior': acc(d > np.quantile(d, 1 - p_yes)), 'oracle(상한)': best}

# 나머지 closed(선택형)는 C-17 의 변형 예측, 기준은 기존 greedy 예측을 쓴다
qf = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/vqarad_pred_closed_qform.jsonl')}
gr = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl')}
n_closed = sum(v['answer_type'] == 'CLOSED' for v in ref.values())
other_ok = sum(ref[str(i)]['answer'].strip().lower() in norm(t) for i, t in qf.items())
base_ok = sum(ref[str(i)]['answer'].strip().lower() in norm(gr[i]) for i in base_ids)
print(f'\nyes/no 형태 {len(base_ids)}문항 중 기준(접미어 [yes, no], greedy) 정답: {base_ok}개 (정답이 yes/no 인 문항 {int(isyn.sum())}개)')
print(f'{"입력":6}{"규칙":14}{"yes/no 정답수":>14}{"closed(272)":>14}')
for name, rr in rules.items():
    for rule, n in rr.items():
        print(f'{name:6}{rule:14}{n:>14}{100 * (n + other_ok) / n_closed:>13.2f}%')
print(f'(기준 closed = {100 * (base_ok + other_ok) / n_closed:.2f}%, 선택형 {len(qf)}문항 정답 {other_ok}개 포함)')
print('※ test 기반 탐색입니다. prior 규칙은 test 정답을 쓰지 않지만 선택지 순서·프롬프트를 test 로 골랐으므로 규칙 준수 행과 별도로 표기하세요. oracle 은 상한 참고용이며 보고하지 마세요.')


## 부록 B — closed(yes/no) 편향 보정 시도 (C-20): **효과 없음**

모델 가중치는 그대로 두고 closed 판정만 yes/no 로그확률 임계값으로 바꿔 보았다(임계값은 val 로만 결정, test 는 한 번). 규칙 준수 체크포인트(`final_proj`, train 만 학습)에서의 결과(2026-11-13 실행):

| | 재판정 대상 정답 | test closed | test overall |
|---|---|---|---|
| 보정 없음(기존) | 279 / 355 | 75.72 | 79.40 |
| 보정 후(val 임계값) | 277 / 355 | 75.24 | 79.21 |

정확도가 임계값에 거의 민감하지 않으므로 SLAKE closed 의 격차는 yes/no 편향이 아니라 정답 판단 자체의 정확도 문제다. 보고 행으로 쓰지 않고 "시도했으나 개선 없음"으로 기록한다. 이 셀은 `final_proj`(train 만 학습) 체크포인트에서만 실행한다 (`trainval` 체크포인트는 자동으로 중단된다).


In [ ]:
# C-20. SLAKE closed(yes/no) 편향 보정 — 규칙 준수 버전: 임계값은 val 로만 정하고, test 는 정해진 규칙을 한 번 적용해 읽는다. (C-10 으로 best 추론이 끝난 뒤에 실행)
# 모델이 "Assistant:" 다음에 내는 첫 답 토큰의 yes/no 로그확률 차이 d = log P(yes) - log P(no) 를 구한다.
# 선택지 순서를 [yes, no] / [no, yes] 로 바꾼 두 번을 물어 평균한다 (순서 편향 완화). 규칙 2가지를 비교하고, 미리 정해 둔 "val" 규칙을 보고값으로 쓴다:
#   0     : d > 0 이면 yes (보정 없음)
#   prior : 예측 yes 비율이 train 의 yes 비율과 같아지도록 임계값을 정한다 (정답 사용 안 함)
#   val   : val 정확도가 최대인 임계값 (보고값. test 정답은 임계값에 쓰지 않는다)
# test 정답으로 고른 oracle 임계값은 계산하지 않는다 (규칙 위반 방지).
import json, re, os, subprocess, time
import numpy as np
from collections import Counter

assert 'SEL' in globals(), '먼저 C-9 로 학습률·epoch 를 선택하세요'
assert 'trainval' not in SEL['best_checkpoint'], 'best 체크포인트가 train+val 로 학습되었습니다 — val 이 학습에 쓰여 임계값을 val 로 정할 수 없습니다 (train 만 학습한 체크포인트로 실행하세요)'
assert os.path.exists(f'{EVAL_DIR}/slake_test.json') and os.path.exists(f'{DRIVE_ROOT}/slake_qlora_best_pred.jsonl'), '먼저 C-9b, C-10 으로 best 의 test 추론을 끝내세요'
print('선택된 체크포인트:', SEL['best_checkpoint'])
print('※ 기준 예측 slake_qlora_best_pred.jsonl 은 같은 체크포인트로 만든 것이어야 합니다 (C-10 을 다른 설정으로 다시 돌렸다면 덮어써졌을 수 있음). FINAL_TRAINVAL=False 로 학습한 규칙 준수 체크포인트에서 실행하세요.')
SUF_A = CLOSED_SUFFIX
SUF_B = ' Please choose from the following two options: [no, yes]'
OFF_TEST = 1000000      # id 규칙: val = i, test = OFF_TEST + i. 순서 B 는 위 id 에 OFF_B 를 더한다 (겹치지 않게 충분히 크게)
OFF_B = 5000000

# 1) 질문 파일: val + test 의 closed 문항 x 2가지 순서
val_rows = json.load(open(f'{QDATA}/slake_val.json'))
test_q = json.load(open(f'{EVAL_DIR}/slake_test.json')); test_ref = {int(k): v for k, v in json.load(open(f'{EVAL_DIR}/slake_ref.json')).items()}
items = []     # (id, split, 이미지, 질문(접미어 제외), 정답)
for i, r in enumerate(val_rows):
    if r['answer_type'] == 'CLOSED':
        items.append((i, 'val', r['image'], r['question'][:-len(SUF_A)] if r['question'].endswith(SUF_A) else r['question'], r['answer']))
for q in test_q:
    if test_ref[q['id']]['answer_type'] == 'CLOSED':
        v = q['conversations'][0]['value']
        items.append((OFF_TEST + q['id'], 'test', q['image'], v[:-len(SUF_A)] if v.endswith(SUF_A) else v, test_ref[q['id']]['answer']))
ynq = []
for (i, sp, img, qq, ans) in items:
    for k, suf in enumerate((SUF_A, SUF_B)):
        ynq.append({'id': i + OFF_B * k, 'image': img, 'conversations': [{'from': 'human', 'value': qq + suf}, {'from': 'gpt', 'value': str(ans)}]})
json.dump(ynq, open(f'{EVAL_DIR}/slake_yn2.json', 'w'))
print(f"closed 문항 val {sum(x[1] == 'val' for x in items)}개 + test {sum(x[1] == 'test' for x in items)}개 x 2가지 순서 = {len(ynq)}회 추론")

# 2) 병합 모델 (C-10 에서 만든 것이 지워졌으면 다시 병합) + 추론 스크립트 복사본 (greedy + 첫 답 토큰의 yes/no 로그확률 기록)
MODEL_DIR = '/content/llava-med-7b-slake-qlora-yn'      # C-10 의 병합 폴더(-best)와 분리: 어떤 체크포인트로 만든 것인지 표시 파일로 확인한다
_marker = f'{MODEL_DIR}/.merged_from'
if not (os.path.exists(f'{MODEL_DIR}/config.json') and os.path.exists(_marker) and open(_marker).read() == SEL['best_checkpoint']):
    sh(f'rm -rf {MODEL_DIR}')
    sh(f'cd /content/LLaVA-Med && {QPY} merge_qlora.py --base {INSTRUCT_MODEL} --adapter {SEL["best_checkpoint"]} --out {MODEL_DIR}')
    open(_marker, 'w').write(SEL['best_checkpoint'])
print('병합 모델 체크포인트:', open(_marker).read())
_src = open('/content/LLaVA-Med/llava/eval/model_vqa_med.py').read()
_pat = re.compile(r"with torch\.inference_mode\(\):\s*output_ids = model\.generate\(.*?stopping_criteria=\[stopping_criteria\]\)\n", re.S)
_n = len(_pat.findall(_src))
if _n == 0:
    _k = _src.find('model.generate')
    print(_src[max(0, _k - 200):_k + 500] if _k >= 0 else 'model.generate 가 없습니다')
assert _n >= 1, 'model_vqa_med.py 의 generate 블록을 찾지 못했습니다 (위 출력을 알려 주세요)'
_new = '''with torch.inference_mode():
            _go = model.generate(input_ids, images=images, do_sample=False, max_new_tokens=24,
                                 stopping_criteria=[stopping_criteria], output_scores=True, return_dict_in_generate=True)
        output_ids = _go.sequences
        _gen = output_ids[0, input_ids.shape[1]:]
        _t = 0
        for _k in range(len(_gen)):
            if ':' in tokenizer.decode(_gen[:_k + 1], skip_special_tokens=True):
                _t = _k + 1
                break
        _vy = [tokenizer.convert_tokens_to_ids(w) for w in ('yes', 'Yes', chr(9601) + 'yes', chr(9601) + 'Yes')]
        _vn = [tokenizer.convert_tokens_to_ids(w) for w in ('no', 'No', chr(9601) + 'no', chr(9601) + 'No')]
        _vy = [x for x in _vy if x is not None and x != tokenizer.unk_token_id]
        _vn = [x for x in _vn if x is not None and x != tokenizer.unk_token_id]
        _ly = _ln = None
        if _t < len(_go.scores):
            _lp = torch.log_softmax(_go.scores[_t][0].float(), -1)
            _ly = torch.logsumexp(_lp[_vy], 0).item()
            _ln = torch.logsumexp(_lp[_vn], 0).item()
        open(args.answers_file + '.yn', 'a').write(json.dumps({'question_id': idx, 'lp_yes': _ly, 'lp_no': _ln, 't': _t,
                                                              'head': tokenizer.decode(_gen[:_t + 3], skip_special_tokens=True)}) + chr(10))
'''
open('/content/LLaVA-Med/llava/eval/model_vqa_yn.py', 'w').write(_pat.sub(lambda m: _new, _src, count=1))

_out = f'{DRIVE_ROOT}/slake_pred_yn2.jsonl'
_side = _out + '.ckpt'                  # 이전에 기록한 로그확률이 다른 체크포인트의 것이면 버린다
if os.path.exists(_side) and open(_side).read() != SEL['best_checkpoint']:
    print('이전 로그확률은 다른 체크포인트의 것입니다 -> 삭제하고 다시 계산합니다')
    for f in (_out, _out + '.yn', _side):
        if os.path.exists(f): os.remove(f)
open(_side, 'w').write(SEL['best_checkpoint'])
_done = set()
if os.path.exists(_out + '.yn'):      # 끊겼으면 이어서: 이미 기록된 문항은 건너뛴다
    _done = {json.loads(l)['question_id'] for l in open(_out + '.yn')}
_todo = [q for q in ynq if q['id'] not in _done]
if _todo:
    json.dump(_todo, open(f'{EVAL_DIR}/slake_yn2_todo.json', 'w'))
    _cmd = (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_yn.py --model-name {MODEL_DIR} '
            f'--question-file {EVAL_DIR}/slake_yn2_todo.json --image-folder {SLAKE_IMG_DIR} --answers-file {_out} > /content/slake_yn2.log 2>&1')
    _proc = subprocess.Popen(['bash', '-c', _cmd]); _t0 = time.time()
    while _proc.poll() is None:
        _c = sum(1 for _ in open(_out + '.yn')) if os.path.exists(_out + '.yn') else 0
        print(f'[{(time.time() - _t0) / 60:5.1f}분] 진행 {_c}/{len(ynq)}', flush=True); time.sleep(60)
    assert _proc.returncode == 0, f'추론 실패 (종료 코드 {_proc.returncode}) — /content/slake_yn2.log 의 끝부분을 확인하세요'
    print(f'추론 완료 ({(time.time() - _t0) / 60:.1f}분)')
else:
    print('이미 모든 문항의 로그확률이 있습니다 — 추론을 건너뜁니다')

CALIB_SPEC = """[보정 방식 명세 — 이 점수를 보고할 때 이 문단을 같이 적는다]
1. 대상: 모델이 greedy 로 yes/no 로 답한 closed 문항(재판정 대상). 모델 출력만으로 정하며 정답은 보지 않는다. 그 외 closed(선택지가 yes/no 가 아닌 문항)와 open 문항은 기존 예측과 기존 채점(closed 는 정답 단어 포함, open 은 토큰 recall) 그대로.
2. 모델 점수: 같은 질문을 선택지 순서 [yes, no] 와 [no, yes] 로 두 번 묻고(greedy), 각각 'Assistant:' 다음 첫 답 토큰에서
   d = log P(yes 계열 토큰) - log P(no 계열 토큰)  (yes/Yes/▁yes/▁Yes 와 no/No/▁no/▁No 의 logsumexp) 를 구해 두 값을 평균한다.
3. 판정: d > 임계값 이면 yes, 아니면 no. 생성된 문장은 closed 에서 쓰지 않는다.
4. 임계값: 'val' 규칙(보고값) = SLAKE val 의 재판정 대상 문항에서 정확도가 최대인 임계값(정렬된 d 의 이웃 중간값 후보, 동점이면 0 에 가까운 것). test 정답은 임계값에 쓰지 않는다.
   참고 규칙 'prior' = 예측 yes 비율이 train 의 yes 비율과 같아지는 val d 분위수, '0' = 보정 없음(d>0).
5. 채점: 재판정 대상은 판정이 정답(yes/no)과 같으면 1, 다르거나 정답이 yes/no 가 아니면 0. 그 외 closed 는 기존 채점. overall = (open recall 합 + closed 정답 수) / 전체 문항 수.
6. 모델 가중치·학습은 바꾸지 않았다. 바뀐 것은 closed 의 답 결정 방식(생성 문장 -> yes/no 로그확률 임계값)뿐이다. 논문의 채점은 생성 문장 기반이므로 보정 행은 별도 행으로 표기한다.
7. 한계: 임계값은 val 분포에서 정해 test 에 그대로 적용한다. val 과 test 의 yes 비율이 다르면 효과가 달라질 수 있다."""

# 3) 분석 — 임계값은 val 로만 정한다. 재판정 대상(CALIBRATABLE) = 모델이 greedy 로 yes/no 로 답한 closed 문항 (정답을 보지 않고 모델 출력으로만 정한다).
# 그 외 closed (예: "Which is larger, A or B?" 처럼 선택지가 yes/no 가 아닌 문항)는 기존 생성 답과 기존 채점을 그대로 쓴다.
yn = {}
for l in open(_out + '.yn'):
    r = json.loads(l); yn[r['question_id']] = r
bad = [q['id'] for q in ynq if q['id'] not in yn or yn[q['id']]['lp_yes'] is None]
assert not bad, f'로그확률이 없는 문항 {len(bad)}개 — /content/slake_yn2.log 확인. 예: {bad[:5]}'
print('첫 답 위치 확인 (t: 답 토큰 위치, head: 모델 출력 앞부분):')
for q in ynq[:4]: print('  ', yn[q['id']]['t'], repr(yn[q['id']]['head']))
def _head_yn(i):
    tk = normalize(yn[i]['head'].lower().split('assistant:')[-1])
    return bool(tk) and tk[0] in ('yes', 'no')
D = lambda i: (yn[i]['lp_yes'] - yn[i]['lp_no'] + yn[i + OFF_B]['lp_yes'] - yn[i + OFF_B]['lp_no']) / 2
V = [x for x in items if x[1] == 'val']; T = [x for x in items if x[1] == 'test']
dv = np.array([D(x[0]) for x in V]); gv = np.array([str(x[4]).strip().lower() for x in V]); cv = np.array([_head_yn(x[0]) for x in V])
dt = np.array([D(x[0]) for x in T]); gt = np.array([str(x[4]).strip().lower() for x in T]); ct = np.array([_head_yn(x[0]) for x in T])
CALIBRATABLE = True
acc = lambda d, g, th: int((((d > th) & (g == 'yes')) | ((d <= th) & (g == 'no'))).sum())      # 정답이 yes/no 가 아닌 문항은 항상 오답

tr = json.load(open(f'{QDATA}/slake_train.json'))
tr_yn = [str(r['answer']).strip().lower() for r in tr if r['answer_type'] == 'CLOSED' and str(r['answer']).strip().lower() in ('yes', 'no')]
p_yes = float(np.mean([a == 'yes' for a in tr_yn]))
dvc, gvc = dv[cv], gv[cv]                       # 임계값은 val 의 재판정 대상 문항으로만 정한다
cand = np.sort(np.unique(dvc)); mids = (cand[:-1] + cand[1:]) / 2
th_val = max(mids, key=lambda m: (acc(dvc, gvc, m), -abs(m)))
THS = {'0': 0.0, 'prior': float(np.quantile(dvc, 1 - p_yes)), 'val': float(th_val)}
print(f'\ntrain 의 yes 비율 {p_yes:.3f} | val closed {len(V)}개 중 재판정 대상 {int(cv.sum())}개 (정답 yes/no {int(np.isin(gvc, ["yes", "no"]).sum())}개) | test closed {len(T)}개 중 재판정 대상 {int(ct.sum())}개')
print('주의: prior 규칙의 임계값은 val 재판정 대상의 d 분포 분위수로 정했다 (test 정답 미사용).')

# 4) test 최종 점수: 재판정 대상 closed 는 보정된 yes/no 로, 그 외 closed 는 기존 생성 답으로, open 은 기존 예측 그대로
base_pred = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/slake_qlora_best_pred.jsonl')}
# 점검: 이번 추론의 모델이 기존 예측(slake_qlora_best_pred.jsonl)을 만든 모델과 같은지 — 기존 예측의 yes/no 와 순서 A 의 d>0 판정이 거의 모두 일치해야 한다
def _lab(text):
    tk = normalize(text); y, n = 'yes' in tk, 'no' in tk
    return 'yes' if y and not n else 'no' if n and not y else 'other'
_pairs = [(_lab(base_pred[x[0] - OFF_TEST]), 'yes' if yn[x[0]]['lp_yes'] > yn[x[0]]['lp_no'] else 'no') for x in T]
_cmp = [(a, b) for a, b in _pairs if a != 'other']
baseline_agree = sum(a == b for a, b in _cmp) / max(len(_cmp), 1)
print(f"\n[모델 동일성 점검] 기존 예측과 이번 추론(순서 A)의 yes/no 일치율: {100 * baseline_agree:.1f}% ({len(_cmp)}문항, 기존 예측이 yes/no 가 아닌 {len(_pairs) - len(_cmp)}개 제외)")
print('  → 95% 이상이면 같은 모델로 본다. 많이 낮으면 기존 예측이 다른 체크포인트(예: train+val)로 만들어졌을 수 있으니 이 표의 "기존" 행을 믿지 말 것.')
base = score_jsonl(f'{EVAL_DIR}/slake_ref.json', f'{DRIVE_ROOT}/slake_qlora_best_pred.jsonl')
n_open, n_closed = base['n_open'], base['n_closed']
open_sum = base['open'] / 100 * n_open
paper_slake = {'open': 83.08, 'closed': 85.34, 'overall': 83.97}
qid_t = [x[0] - OFF_TEST for x in T]
other_ok = sum(closed_accuracy(base_pred[q], str(x[4])) for q, x, c in zip(qid_t, T, ct) if not c)       # 재판정 대상이 아닌 closed: 기존 답 그대로
base_cal_ok = sum(closed_accuracy(base_pred[q], str(x[4])) for q, x, c in zip(qid_t, T, ct) if c)          # 재판정 대상의 기존 정답 수
print(f"\n{'규칙':10s}{'임계값':>9s}{'val 정확도(대상)':>17s}{'대상 정답수':>11s}{'test closed':>13s}{'test overall':>14s}{'논문 대비':>10s}")
print(f"{'보정 없음(기존)':8s}{'':>9s}{'':>17s}{base_cal_ok:>8.0f}/{int(ct.sum())}{base['closed']:11.2f}%{base['overall']:14.2f}{base['overall'] - paper_slake['overall']:+10.2f}")
for name, th in THS.items():
    v_ok = acc(dvc, gvc, th); t_ok = acc(dt[ct], gt[ct], th)
    closed = 100 * (t_ok + other_ok) / n_closed; overall = (open_sum + t_ok + other_ok) / (n_open + n_closed) * 100
    print(f"{name:10s}{th:9.3f}{100 * v_ok / max(len(dvc), 1):16.2f}%{t_ok:>8d}/{int(ct.sum())}{closed:11.2f}%{overall:14.2f}{overall - paper_slake['overall']:+10.2f}   {'← 보고값 후보' if name == 'val' else ''}")
print(f"(test: open {n_open}문항 {base['open']:.2f}% 그대로, 재판정 대상이 아닌 closed {int((~ct).sum())}문항은 기존 생성 답 그대로 {other_ok:.0f}개 정답)")
print('※ 규칙 준수: 임계값은 val 로만 정하고 test 는 한 번 읽었다. 보고값은 "val" 행이다. 선택지 순서([yes, no]/[no, yes] 평균)는 val 에서 비교하지 않고 미리 정한 방식이다.')
json.dump({'thresholds': THS, 'p_yes_train': p_yes, 'spec': CALIB_SPEC}, open(f'{QROOT}/yn_calibration.json', 'w'), indent=1)
print(chr(10) + CALIB_SPEC)


### 부록 B-2 — 병합 모델에 학습된 `mm_projector` 가 들어갔는지 확인

`merge_qlora.py` 가 "어댑터에 mm_projector 가중치가 없습니다"라고 출력할 때가 있다. 이 메시지는 키 이름 필터 때문에 잘못 나온 것이다(어댑터에는 `...mm_projector.weight/bias` 가 있다). 실제 반영 여부는 아래 셀로 확인한다. 2026-11-13 확인 결과 최대 차이 0.0 이어서 학습된 프로젝터가 병합 모델에 그대로 들어가 있었다 (병합 스크립트의 필터는 이후 수정함).


In [ ]:
# B-2. 병합 모델의 mm_projector 가 어댑터(학습된 값)와 같은지 확인 — C-20 이나 C-10 으로 병합 모델을 만든 뒤 실행 (GPU 불필요)
import json, torch, os
from safetensors.torch import load_file
assert 'SEL' in globals(), '먼저 C-9 / C-9d 로 체크포인트를 선택하세요'
ck  = SEL['best_checkpoint']
out = globals().get('MODEL_DIR', '/content/llava-med-7b-slake-qlora-yn')       # C-20 의 병합 폴더 (C-10 의 것을 보려면 '/content/llava-med-7b-slake-qlora-best')
ad = load_file(f'{ck}/adapter_model.safetensors') if os.path.exists(f'{ck}/adapter_model.safetensors') else torch.load(f'{ck}/adapter_model.bin', map_location='cpu')
pk = [k for k in ad if 'mm_projector' in k and k.endswith('weight')]
assert pk, '어댑터에 mm_projector 가중치가 없습니다 (프로젝터를 학습하지 않은 체크포인트)'
aw = ad[pk[0]].float()
idx = json.load(open(f'{out}/pytorch_model.bin.index.json'))['weight_map']
key = [k for k in idx if k.endswith('mm_projector.weight')][0]
mw = torch.load(f'{out}/{idx[key]}', map_location='cpu')[key].float()
print('체크포인트:', ck); print('병합 모델 키:', key, '| 어댑터와의 최대 차이:', float((mw - aw).abs().max()), '(0 근처면 학습된 프로젝터가 반영됨)')
